# Attack from the boundary — and mimicry improved by CW

A different attack geometry: instead of pushing a malware sample across the decision boundary from inside, find the **closest benign-side point**, walk to the boundary, and start the attack from there. The second half combines the two ideas — run **mimicry** first, then let **CW improve the mimicry results** — against both AT-rFGSM and the contrastive combined_model, in L2 and L1, with the best confidence values found along the way (0.4 for AT-rFGSM, 0.2 for combined_model) recorded in the runs.

**Where this fits:** this section tests a different defence idea — train the detector with **supervised contrastive learning** so its feature space separates the classes more cleanly, then attack the result with the machinery of `03-attacks/`. The CW machinery aimed at these models continues the lineage of `03-attacks/cw_sine_lineage.ipynb`.

*Consolidated from `mimicry_cw_complete.ipynb` (the developed version) and `mimicry.ipynb` (originally in `CW-CL`). Outputs cleared.*

## Step 1 — Setup and the rounding/evaluation harness

# Load dataset and libraries

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader,WeightedRandomSampler
from sklearn.model_selection import train_test_split
from scipy import sparse
import gdown
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score,balanced_accuracy_score
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import precision_recall_curve, roc_curve, auc
import time
import os
import math
from sklearn.manifold import TSNE
import seaborn as sns
import pandas as pd
import tqdm
import pickle

torch.manual_seed(0)
np.random.seed(0)

# Device configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

In [ ]:
download_links = ['https://github.com/mostafa-ja/mal_adv3/raw/main/drebin/sparse_matrix_0.npz',
                  'https://github.com/mostafa-ja/mal_adv3/raw/main/drebin/sparse_matrix_1.npz',
                  'https://github.com/mostafa-ja/mal_adv3/raw/main/drebin/sparse_matrix_2.npz',
                  'https://github.com/mostafa-ja/mal_adv3/raw/main/drebin/sparse_matrix_y0.npz',
                  'https://github.com/mostafa-ja/mal_adv3/raw/main/drebin/sparse_matrix_y1.npz',
                  'https://github.com/mostafa-ja/mal_adv3/raw/main/drebin/sparse_matrix_y2.npz',
                  'https://github.com/mostafa-ja/mal_adv4/raw/main/dataset/insertion_array.pkl',
                  'https://github.com/mostafa-ja/mal_adv4/raw/main/dataset/removal_array.pkl',
                  'https://github.com/mostafa-ja/mal_adv4/raw/main/dataset/adverserial_attacks_functions.py',
                  'https://github.com/mostafa-ja/mal_adv3/raw/main/drebin/model_AT_rFGSM.pth',
                  'https://github.com/mostafa-ja/mal_adv3/raw/main/drebin/best_model_AT_imax.pth',
                  'https://github.com/mostafa-ja/mal_adv3/raw/main/data/model_AT_max%20(4).pth',
]

output_filepath = '/content/'
for link in download_links:
  gdown.download(link, output_filepath)

In [ ]:
from adverserial_attacks_functions import *

In [ ]:
# Open the .pkl file
with open('/content/insertion_array.pkl', 'rb') as f:
    # Load the object
    insertion_array = pickle.load(f)

# Close the file
f.close()

insertion_array = torch.tensor(insertion_array, dtype=torch.uint8).to(device)
print(len(insertion_array))

# Open the .pkl file
with open('/content/removal_array.pkl', 'rb') as f:
    # Load the object
    removal_array = pickle.load(f)

# Close the file
f.close()

removal_array = torch.tensor(removal_array, dtype=torch.uint8).to(device)
print(len(removal_array))

In [ ]:
# Load dataset
X_train = sparse.load_npz("/content/sparse_matrix_0.npz").toarray()
X_val = sparse.load_npz("/content/sparse_matrix_1.npz").toarray()
X_test = sparse.load_npz("/content/sparse_matrix_2.npz").toarray()

X_train = torch.tensor(X_train, dtype=torch.int8)
X_val = torch.tensor(X_val, dtype=torch.int8)
X_test = torch.tensor(X_test, dtype=torch.int8)


y_train = sparse.load_npz("/content/sparse_matrix_y0.npz").toarray().reshape((-1, 1))
y_val = sparse.load_npz("/content/sparse_matrix_y1.npz").toarray().reshape((-1, 1))
y_test = sparse.load_npz("/content/sparse_matrix_y2.npz").toarray().reshape((-1, 1))

y_train = torch.tensor(y_train, dtype=torch.int8)
y_val = torch.tensor(y_val, dtype=torch.int8)
y_test = torch.tensor(y_test, dtype=torch.int8)


print("Shapes:")
print("x_train:", X_train.shape)
print("x_val:", X_val.shape)
print("x_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_val:", y_val.shape)
print("y_test:", y_test.shape)

In [ ]:
# number of benigns and malicious sample in training dataset
n_ben = (y_train.squeeze()== 0).sum().item()
n_mal = (y_train.squeeze()== 1).sum().item()

# Combine features and labels into datasets
# we use dtype=torch.int8, for Memory-Efficient here, later we will convert to float
train_dataset = TensorDataset(X_train, y_train)
val_dataset = TensorDataset(X_val, y_val)
test_dataset = TensorDataset(X_test, y_test)

In [ ]:
# Create an instance of your model
model_AT_rFGSM = MalwareDetectionModel().to(device)

# Load model parameters
model_AT_rFGSM.load_state_dict(torch.load('model_AT_rFGSM.pth', map_location=torch.device(device)))

# Create an instance of your model
model_AT_imax = MalwareDetectionModel().to(device)

# Load model parameters
model_AT_imax.load_state_dict(torch.load('best_model_AT_imax.pth', map_location=torch.device(device)))

In [ ]:
# Define the Linear Classifier(NO NEED)
class LinearClassifier(nn.Module):
    def __init__(self, dim_in=200, num_classes=2):
        super(LinearClassifier, self).__init__()
        self.fc = nn.Linear(dim_in, num_classes)

    def forward(self, x):
        return self.fc(x)


# Define the CoreModel
class CoreModel(nn.Module):
    def __init__(self):
        super(CoreModel, self).__init__()
        self.fc1 = nn.Linear(10000, 200)
        self.dropout1 = nn.Dropout(0.6)
        self.fc2 = nn.Linear(200, 200)
        self.dropout2 = nn.Dropout(0.6)
        self.fc3 = nn.Linear(200, 2)

    def forward(self, x, return_z=False):
        x = F.relu(self.fc1(x))
        #x = self.dropout1(x)
        z = F.relu(self.fc2(x))
        z2 = self.dropout2(z)
        output  = self.fc3(z2)
        if return_z:
            return output, z

        return output

    def forward_from_z(self, z):
        output = self.fc3(z)
        return output

# Define the Projection Head
class ProjHead(nn.Module):
    def __init__(self, dim_in=200, feat_dim=128):
        super(ProjHead, self).__init__()
        self.head = nn.Linear(dim_in, feat_dim)

    def forward(self, x):
        #x = F.normalize(x, dim=1)
        z_proj = self.head(x)
        z_proj = F.normalize(z_proj, dim=1)
        return z_proj



# Define the Wrapper
class Wrapper(nn.Module):
    def __init__(self, CoreModel, ProjLayer):
        super(Wrapper, self).__init__()
        self.CoreModel = CoreModel
        self.ProjLayer = ProjLayer
        #self.LC = LC

    def forward(self, x, return_z=False):
        output, z = self.CoreModel(x, return_z=True)
        z_proj = self.ProjLayer(z)
        #output = self.LC(z)
        if return_z:
            return output, z_proj
        else:
            return output

# Instantiate the components
core_model = CoreModel()
proj_head = ProjHead(200, 128)
#linear_classifier = LinearClassifier(128, 2)

# Combine them in the Wrapper
combined_model = Wrapper(core_model, proj_head)

# Example usage
x = torch.randn(2, 10000)  # Example input
output, z_proj = combined_model(x,return_z=True)
print(output.shape)
print(z_proj.shape)

In [ ]:
# Device configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

combined_model = Wrapper(core_model, proj_head).to(device)
combined_model.load_state_dict(torch.load('model_AT_max%20(4).pth', map_location=torch.device(device)))

In [ ]:
def pgd_min22(x, y, model, insertion_array, removal_array, k=25, step_length=0.02, norm='linf', confidence=0.1, initial_rounding_threshold=0.5, round_threshold=0.5, random=False, is_report_loss_diff=True, is_sample=False):
    """
    Projected Gradient Descent (PGD) adversarial attack (loss based on goal's class, which we have to minimize the loss).
    :param y: Ground truth labels
    :param x: Feature vector
    :param model: Neural network model
    :param k: Number of steps
    :param step_length: Step size for each iteration
    :param norm: Norm used for perturbation ('linf' or 'l2')
    :param initial_rounding_threshold: Threshold parameter for rounding the initial x_next
    :param round_threshold: Threshold parameter for rounding
    :param random: Flag to generate random thresholds
    :param is_report_loss_diff: Flag to report loss difference
    :param is_sample: Flag to sample randomly from the feasible area
    :return: The adversarial version of x (tensor)
    """
    model.eval()

    # Compute natural loss
    criterion = nn.CrossEntropyLoss(reduction='none')
    loss_natural = criterion(model(x), torch.zeros_like(y.view(-1).long()))

    # Initialize starting point
    x_next = x.clone()
    #x_next = get_x0(x_next, initial_rounding_threshold, is_sample)


    # Expand insertion_array and removal_array to match the batch size
    expanded_insertion_array = insertion_array.expand(x.shape[0], -1)
    expanded_removal_array = removal_array.expand(x.shape[0], -1)

    # Update insertion and removal arrays based on input x
    insertion_array_updated = torch.bitwise_or(expanded_insertion_array, x.to(torch.uint8))
    removal_array_updated = torch.bitwise_or(expanded_removal_array, 1 - x.to(torch.uint8))

    # Multi-step PGD
    for t in range(k):
        #print('*********** ',t)
        # Forward pass
        x_var = x_next.clone().detach().requires_grad_(True)
        y_model = model(x_var)
        #loss = criterion(y_model, torch.zeros_like(y.view(-1).long()))
        #print(loss)
        #print('loss_mal : ',criterion(y_model, torch.zeros_like(y.view(-1).long())))

        batch_size = x_var.shape[0]
        real = y_model[torch.arange(batch_size), y.view(-1).long()]
        other = y_model[torch.arange(batch_size), (y.view(-1).long()- 1)]
        loss = torch.clamp((real - other + confidence), min=0.0)
        #loss = (real - other)


        # Compute gradient
        grad_vars = torch.autograd.grad(loss.mean(), x_var)
        gradients = -(grad_vars[0].data)
        #print('torch.abs(gradients).sum() : ',torch.abs(gradients).sum(dim=-1))

        #grad4insertion = (gradients >= 0) *(x_var < 1.)* insertion_array_updated * gradients
        grad4removal = (gradients < 0) * (x_var > 0.) * removal_array_updated * gradients

        grad4insertion = (gradients >= 0) *(x_var < 1.)* gradients
        #grad4removal = (gradients < 0) * (x_var > 0.) * gradients

        gradients = grad4removal + grad4insertion
        #print(torch.abs(gradients).sum())

        # Norm
        if norm == 'linf':
            perturbation = torch.sign(gradients).float()

        elif norm == 'free':
            perturbation = gradients

        elif norm == 'max':
            max_values, _ = torch.max(torch.abs(gradients), dim=-1, keepdim=True)
            perturbation = (gradients / (max_values+ 1e-20))

        elif norm == 'l2':
            l2norm = torch.linalg.norm(gradients, dim=-1, keepdim=True)
            perturbation = (gradients / (l2norm + 1e-20)).float()

        elif norm == 'l1':
            # consider just features of a sample which are not updated yet(because our update is 0to1 or 1to0 not stepwise)
            un_mod = torch.abs(x - x_var) <= 1e-6
            gradients = gradients * un_mod
            max_grad = torch.topk(torch.abs(gradients).view(gradients.size(0), -1), 1, dim=-1)[0]
            #print('max_grad ',max_grad)
            perturbation = (torch.abs(gradients) >= max_grad).float() * torch.sign(gradients).float()
            done = get_done(x_next, y, model)
            if torch.all(done):
                break
            perturbation[done] = 0.

        else:
            raise ValueError("Expect 'l1' or 'l2' or 'linf' norm.")

        # Update x_next
        x_next = torch.clamp(x_next + perturbation * step_length, min=0., max=1.)

    # Rounding step
    if random:
       round_threshold = torch.rand(x_next.size())
    #x_next = round_x(x_next, round_threshold=round_threshold)

    # Compute adversarial loss
    #loss_adv = criterion(model(x_next), torch.zeros_like(y.view(-1).long())).data

    # Replace with natural if adversarial loss is higher
    #replace_flag = (loss_adv > loss_natural).squeeze()
    #x_next[replace_flag] = x[replace_flag]

    diff = torch.abs(x_next - x).sum(dim=-1)

    if is_report_loss_diff:
        done = get_done(x_next, y, model)
        print(f"PGD {norm}: Attack effectiveness {done.sum().item() / x.size()[0] * 100:.3f}%.")
        print('mean of differnce featres',torch.mean(diff))

    return x_next

In [ ]:
# Define the DataLoader for training, validation, and test sets
batch_size = 100000
#sampler = weight_sampler(y_train)
#train_loader = DataLoader(train_dataset, batch_size=batch_size,sampler=sampler)
train_loader = DataLoader(train_dataset, batch_size=batch_size,shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

In [ ]:
def adv_predict(test_loader, model, attack, device, **kwargs):

    model.eval()
    n_samples = 0
    cor_test = 0
    cor_ad_test = 0

    with torch.no_grad():
        for x_test, y_test in test_loader:
            x_test, y_test = x_test.to(torch.float32).to(device), y_test.to(device)
            #outputs = model(x_test)
            #predicted = outputs.argmax(1).unsqueeze(1)
            #acc_test = (predicted == y_test).sum().item() / len(y_test)
            #avg_acc_test.append(acc_test)

            mal_x_batch, mal_y_batch = x_test[y_test.squeeze() == 1], y_test[y_test.squeeze() == 1]
            n_samples += len(mal_y_batch)

            outputs = model(mal_x_batch)
            predicted = outputs.argmax(1)
            cor_test += (predicted == 1).sum().item()

            # Generate adversarial examples for test set
            with torch.enable_grad():
                pertb_mal_x= attack(mal_x_batch, mal_y_batch, model, **kwargs)

            outputs = model(pertb_mal_x)
            y_pred = outputs.argmax(1)
            cor_ad_test += (y_pred == 1).sum().item()

            #print(outputs)
            print(len(mal_y_batch))
            print((y_pred == 0).sum().item())
            #print((y_pred == 0))
            #print('mean difference ',(torch.abs(pertb_mal_x - mal_x_batch)).sum()/len(mal_y_batch))
            #print((torch.abs(pertb_mal_x[y_pred == 0] - mal_x_batch[y_pred == 0])).sum())
            #print((torch.abs(pertb_mal_x - mal_x_batch)).sum(1))

            #print('***************************')


    # Calculate test accuracy
    print(f"Accuracy of just malwares (without attack): {(cor_test / n_samples) * 100:.4}% | Under attack: {(cor_ad_test / n_samples) * 100:.4}%.")

    return pertb_mal_x

In [ ]:
def sum_of_top_n_differences(adv_rounded, mal_x_batch, y_pred, n):
    """
    Calculate the sum of the top n smallest differences between adv_rounded and mal_x_batch
    for samples where y_pred == 0.

    Parameters:
    adv_rounded (torch.Tensor): The rounded adversarial examples tensor.
    mal_x_batch (torch.Tensor): The original malicious examples tensor.
    y_pred (torch.Tensor): The prediction tensor.
    n (int): The number of top differences to sum.

    Returns:
    torch.Tensor: The sum of the top n smallest differences.
    """
    # Step 1: Calculate the total differences
    total_difference = torch.abs(adv_rounded[y_pred == 0] - mal_x_batch[y_pred == 0]).sum(1)

    # Step 2: Sort the differences in ascending order
    sorted_differences, _ = torch.sort(total_difference)

    # Step 3: Select the top n least differences
    top_n_least_differences = sorted_differences[:n]

    # Step 4: Sum these n differences
    sum_top_n_least_differences = top_n_least_differences.sum()

    return sum_top_n_least_differences

In [ ]:
def evaluate_non_rounded_adversarial_examples(mal_x_batch, mal_y_batch, adv, combined_model):
    """
    Evaluate the model's performance on non-rounded adversarial examples.

    Parameters:
    - mal_x_batch: Original malware input batch
    - mal_y_batch: Ground truth labels for malware batch
    - adv: Adversarial examples generated from mal_x_batch
    - combined_model: The trained model being evaluated

    Returns:
    - A dictionary containing evaluation metrics
    """

    # Set the model to evaluation mode
    combined_model.eval()

    # Get the model's predictions on adversarial examples
    outputs = combined_model(adv)
    y_pred = outputs.argmax(dim=1)

    # Calculate metrics
    num_malwares = len(mal_y_batch)
    num_successful_attacks = (y_pred == 0).sum().item()
    total_difference = (torch.abs(adv[y_pred == 0] - mal_x_batch[y_pred == 0])).sum().item()
    total_difference_TOP100 = sum_of_top_n_differences(adv, mal_x_batch, y_pred, 100)
    accuracy = (y_pred == 1).sum().item() / num_malwares * 100

    # Print the evaluation results
    print(f"Evaluation Results on NOT_Rounded Adversarial Examples")
    print(f"-----------------------------------------------")
    print(f"Number of malware samples: {num_malwares}")
    print(f"Number of successful attacks: {num_successful_attacks}")
    print(f"Total difference (L1 norm) between successful adversarial examples and original: {total_difference}")
    print(f"Total L1 norm difference - TOP 100 (successful adversarial examples vs. original): {total_difference_TOP100}")
    print(f"Accuracy of the model on malwares under attack: {accuracy:.2f}%")

    # Return results as a dictionary
    return {
        "num_malwares": num_malwares,
        "num_successful_attacks": num_successful_attacks,
        "total_difference": total_difference,
        "total_difference_TOP100": total_difference_TOP100,
        "accuracy": accuracy
    }

In [ ]:
def evaluate_rounded_adversarial_examples(mal_x_batch, mal_y_batch, adv, combined_model, removal_array):
    """
    Evaluate the model's performance on rounded adversarial examples.

    Parameters:
    - mal_x_batch: Original malware input batch
    - mal_y_batch: Ground truth labels for malware batch
    - adv: Adversarial examples generated from mal_x_batch
    - combined_model: The trained model being evaluated
    - removal_array: Array used in the final rounding step

    Returns:
    - A dictionary containing evaluation metrics
    """

    # Set the model to evaluation mode
    combined_model.eval()

    # Apply the final rounding solution to adversarial examples
    adv_rounded = final_rounding(mal_x_batch, adv, combined_model, removal_array)

    # Get the model's predictions on the rounded adversarial examples
    outputs = combined_model(adv_rounded)
    y_pred = outputs.argmax(dim=1)

    # Calculate evaluation metrics
    num_malwares = len(mal_y_batch)
    num_successful_attacks = (y_pred == 0).sum().item()
    total_difference = (torch.abs(adv_rounded[y_pred == 0] - mal_x_batch[y_pred == 0])).sum().item()
    total_difference_TOP100 = sum_of_top_n_differences(adv_rounded, mal_x_batch, y_pred, 100)
    total_difference_TOP150 = sum_of_top_n_differences(adv_rounded, mal_x_batch, y_pred, 150)
    total_difference_TOP200 = sum_of_top_n_differences(adv_rounded, mal_x_batch, y_pred, 200)
    accuracy = (y_pred == 1).sum().item() / num_malwares * 100

    # Print the evaluation results
    print(f"Evaluation Results on Rounded Adversarial Examples")
    print(f"-----------------------------------------------")
    print(f"Number of malware samples: {num_malwares}")
    print(f"Number of successful attacks: {num_successful_attacks}")
    print(f"Total L1 norm difference (successful adversarial examples vs. original): {total_difference}")
    print(f"Total L1 norm difference - TOP 100 (successful adversarial examples vs. original): {total_difference_TOP100}")
    print(f"Total L1 norm difference - TOP 150 (successful adversarial examples vs. original): {total_difference_TOP150}")
    print(f"Total L1 norm difference - TOP 200 (successful adversarial examples vs. original): {total_difference_TOP200}")
    print(f"Accuracy of the model on malware under attack: {accuracy:.2f}%")

    # Return results as a dictionary
    return {
        "adv_rounded": adv_rounded,
        "num_malwares": num_malwares,
        "num_successful_attacks": num_successful_attacks,
        "total_difference": total_difference,
        "total_difference_TOP100": total_difference_TOP100,
        "total_difference_TOP150": total_difference_TOP150,
        "total_difference_TOP200": total_difference_TOP200,
        "accuracy": accuracy
    }

In [ ]:
def ensure_fixed_elements_unchanged(mal_x_batch, adv, adv_rounded, removal_array):
    """
    Ensure that fixed elements are not changed and calculate differences between adversarial examples and the original batch.

    Parameters:
    - mal_x_batch: Original malware input batch
    - adv: Adversarial examples generated from mal_x_batch
    - adv_rounded: Rounded adversarial examples
    - removal_array: Array indicating the fixed features

    Returns:
    - A dictionary containing differences for fixed and non-fixed features
    """

    # Ensure `removal_array` is expanded to match the batch size
    expanded_removal_array = removal_array.expand(mal_x_batch.shape[0], -1)

    # Create a mask where non-fixed features are set to 1
    non_fixed_features_mask = torch.bitwise_or(expanded_removal_array, 1 - mal_x_batch.to(torch.uint8))

    # Calculate indices where the mask is 0 (fixed) or 1 (non-fixed)
    mask_fixed = non_fixed_features_mask == 0
    mask_non_fixed = non_fixed_features_mask == 1

    # Calculate the differences for fixed and non-fixed features
    diff_adv_fixed = (torch.abs(adv - mal_x_batch) * mask_fixed).sum().item()
    diff_adv_non_fixed = (torch.abs(adv - mal_x_batch) * mask_non_fixed).sum().item()
    diff_adv_rounded_fixed = (torch.abs(adv_rounded - mal_x_batch) * mask_fixed).sum().item()
    diff_adv_rounded_non_fixed = (torch.abs(adv_rounded - mal_x_batch) * mask_non_fixed).sum().item()

    # Print the differences
    print(f"Differences when mask is 0 (fixed features):")
    print(f"  Adv vs. Original: {diff_adv_fixed}")
    print(f"  Rounded Adv vs. Original: {diff_adv_rounded_fixed}")
    print(f"Differences when mask is 1 (non-fixed features):")
    print(f"  Adv vs. Original: {diff_adv_non_fixed}")
    print(f"  Rounded Adv vs. Original: {diff_adv_rounded_non_fixed}")

    # Return the results as a dictionary
    return {
        "diff_adv_fixed": diff_adv_fixed,
        "diff_adv_non_fixed": diff_adv_non_fixed,
        "diff_adv_rounded_fixed": diff_adv_rounded_fixed,
        "diff_adv_rounded_non_fixed": diff_adv_rounded_non_fixed
    }

In [ ]:
def compare_round(mal_x_batch,newimg, model,removal_array):

    adv_rounded = final_rounding(mal_x_batch, newimg, model,removal_array)
    #l0dist = torch.sum((mal_x_batch - newimg)!=0, dim=1)
    l0dist = torch.sum(torch.abs(adv_rounded - mal_x_batch), dim=1)
    logits = model(adv_rounded)
    output = torch.argmax(logits, dim=1)

    return output == 0, l0dist

In [ ]:
def adv_predict_rounding(test_loader, model, attack, device, **kwargs):

    model.eval()
    n_samples = 0
    cor_test = 0
    cor_ad_test = 0

    with torch.no_grad():
        for x_test, y_test in test_loader:
            x_test, y_test = x_test.to(torch.float32).to(device), y_test.to(device)
            #outputs = model(x_test)
            #predicted = outputs.argmax(1).unsqueeze(1)
            #acc_test = (predicted == y_test).sum().item() / len(y_test)
            #avg_acc_test.append(acc_test)

            mal_x_batch, mal_y_batch = x_test[y_test.squeeze() == 1], y_test[y_test.squeeze() == 1]
            n_samples += len(mal_y_batch)

            outputs = model(mal_x_batch)
            predicted = outputs.argmax(1)
            cor_test += (predicted == 1).sum().item()

            # Generate adversarial examples for test set
            with torch.enable_grad():
                pertb_mal_x= attack(mal_x_batch, mal_y_batch, model, **kwargs)

            adv_rounded = final_rounding(mal_x_batch, pertb_mal_x, model,removal_array)

            outputs = model(adv_rounded)
            y_pred = outputs.argmax(1)
            cor_ad_test += (y_pred == 1).sum().item()

            #print(outputs)
            print(len(mal_y_batch))
            print((y_pred == 0).sum().item())
            #print((y_pred == 0))
            #print('mean difference ',(torch.abs(pertb_mal_x - mal_x_batch)).sum()/len(mal_y_batch))
            #print((torch.abs(pertb_mal_x[y_pred == 0] - mal_x_batch[y_pred == 0])).sum())
            #print((torch.abs(pertb_mal_x - mal_x_batch)).sum(1))

            #print('***************************')


    # Calculate test accuracy
    print(f"Accuracy of just malwares (without attack): {(cor_test / n_samples) * 100:.4}% | Under attack: {(cor_ad_test / n_samples) * 100:.4}%.")

    return pertb_mal_x

In [ ]:
def final_rounding(oimgs, newimg, model, removal_array):
    # Expand removal array to match the batch size
    non_fixed_features_mask = torch.bitwise_or(removal_array.expand(oimgs.shape[0], -1), 1 - oimgs.to(torch.uint8)).to(device)

    # Ensure all tensors are on the correct device
    oimgs, perturbed = oimgs.to(device), newimg.to(device)

    # Set the model to evaluation mode
    model.eval()

    # Calculate predictions and significant changes efficiently using GPU
    with torch.no_grad():
        predictions = model(oimgs).argmax(dim=1).to(device)
        active_mask = predictions == 1  # Efficiently create active mask on GPU
        total_change = (perturbed - oimgs).abs().to(device)  # Calculate absolute difference on GPU
        significant_mask = total_change > 0.1  # Determine significant changes on GPU
        sorted_idx = total_change.argsort(dim=1, descending=True)  # Sort on GPU
        changes = (perturbed > oimgs).float().to(device)

    # Copy original images to apply final rounding
    final = oimgs.clone().to(device)

    # Iterate up to the maximum number of significant changes
    # practicaly we dont have perturbs more than 120 !!
    max_significant_changes = min(significant_mask.sum(dim=1).max().item(), 120)

    for i in range(max_significant_changes):
        # Safeguard against empty active mask
        if not active_mask.any():
            break

        # Get the indices of active samples
        active_indices = torch.where(active_mask)[0].to(device)

        # Get the relevant indices for current change
        indices = sorted_idx[active_indices, i]

        # since max_significant_changes is relatively large for some samples, possible that zero changes which belong to fixed_features get updated
        final[active_indices, indices] = torch.where(non_fixed_features_mask[active_indices, indices],
                                                     changes[active_indices, indices], final[active_indices, indices])

        # Update the active_mask based on the model's prediction
        with torch.no_grad():
            active_mask[active_indices] = (model(final[active_indices]).argmax(dim=1) == 1)

    return final

In [ ]:
for x_test, y_test in test_loader:
    x_test, y_test = x_test.to(torch.float32).to(device), y_test.to(device)
    mal_x_batch, mal_y_batch = x_test[y_test.squeeze() == 1], y_test[y_test.squeeze() == 1]
    ben_x_batch, ben_y_batch = x_test[y_test.squeeze() == 0], y_test[y_test.squeeze() == 0]
    print('mal_x_batch ',mal_x_batch.shape)
    print('ben_x_batch ',ben_x_batch.shape)
    break

## Step 2 — Attack from the boundary: find the border point, then step

# Attack from boundary

In [ ]:
clean_x = mal_x_batch[:1].clone()
clean_x.shape

In [ ]:
# Expand insertion_array and removal_array to match the batch size
expanded_removal_array = removal_array.expand(clean_x.shape[0], -1)

non_removal_array_updated = torch.bitwise_and(1 - expanded_removal_array.to(torch.uint8), clean_x.to(torch.uint8))

In [ ]:
closest_distance = float('inf')
starting_point = None

with torch.no_grad():
    for ben in ben_x_batch:
        input = torch.bitwise_or(non_removal_array_updated,ben.to(torch.uint8)).to(torch.float32)


        # Calculate the L2 distance between the clean input and the misclassified sample
        distance = (torch.abs(clean_x - input)).sum()

        # Update the starting point if this is the closest misclassified sample
        if distance < closest_distance:
            outputs = model_AT_rFGSM(input)
            if outputs.argmax() == 0:
                closest_distance = distance
                starting_point = input.clone()
                print(closest_distance)

In [ ]:
model_AT_rFGSM(starting_point)

In [ ]:
sample = (clean_x + starting_point)/2.
model_AT_rFGSM(sample)

In [ ]:
point_a = starting_point.clone()
point_b = clean_x.clone()

closest_distance = (torch.abs(point_a - clean_x)).sum()
print(closest_distance)
border_point = point_a

model_AT_rFGSM.eval()
for i in range(20):
  sample = (point_a + point_b)/2.
  if model_AT_rFGSM(sample).argmax() == 0:
    border_point = sample
    point_a = sample
    closest_distance = (torch.abs(point_a - clean_x)).sum()
    print(closest_distance)
  else:
    point_b = sample

In [ ]:
model_AT_rFGSM(border_point)

In [ ]:
step_length = 0.1

x_next = border_point.clone().detach()
x_var = x_next.clone().detach().requires_grad_(True)
y_model = model_AT_rFGSM(x_var)
mal = y_model[:, 1]
ben = y_model[:, 0]
loss = (mal - ben)
print(loss)

# Compute gradient
grad_vars = torch.autograd.grad(loss.mean(), x_var)
gradients = -(grad_vars[0].data)


l2norm = torch.linalg.norm(gradients, dim=-1, keepdim=True)
perturbation = (gradients / (l2norm + 1e-20)).float()

x_next = torch.clamp(x_next + perturbation * step_length, min=0., max=1.)

In [ ]:
model_AT_rFGSM(x_next)

In [ ]:
y_model = model_AT_rFGSM(x_next)
real = y_model[:, 1]
other = y_model[:, 0]
loss = (real - other)
print(loss)

In [ ]:
model_AT_rFGSM.eval()

x_next = border_point.clone().detach()
x_var = x_next.clone().detach().requires_grad_(True)
y_model = model_AT_rFGSM(x_var)
mal = y_model[:, 1]
ben = y_model[:, 0]
loss = (mal - ben)
#print(loss)

# Compute gradient
grad_vars = torch.autograd.grad(loss.mean(), x_var)
gradients = -(grad_vars[0].data)


l2norm = torch.linalg.norm(gradients, dim=-1, keepdim=True)
f_norm = (gradients / (l2norm + 1e-20)).float()

In [ ]:
perturbation = delta_norm - (delta_norm * f_norm)
perturbation[0][:100]

In [ ]:
step_length = 0.001
x_next = torch.clamp(x_next + perturbation * step_length, min=0., max=1.)
y_model = model_AT_rFGSM(x_next)
real = y_model[:, 1]
other = y_model[:, 0]
loss = (real - other)
print(loss)

In [ ]:
x_var = x_next.clone().detach().requires_grad_(True)
y_model = model_AT_rFGSM(x_var)
mal = y_model[:, 1]
ben = y_model[:, 0]

loss = (mal - ben)
print(loss)

# Compute gradient
grad_vars = torch.autograd.grad(loss.mean(), x_var)
gradients = -(grad_vars[0].data)


l2norm = torch.linalg.norm(gradients, dim=-1, keepdim=True)
f_norm = (gradients / (l2norm + 1e-20)).float()

In [ ]:
perturbation2 = -loss * f_norm
x_next = torch.clamp(x_next + perturbation2 * step_length, min=0., max=1.)
y_model = model_AT_rFGSM(x_next)
real = y_model[:, 1]
other = y_model[:, 0]
loss = (real - other)
print(loss)

In [ ]:
removal_array_updated = torch.bitwise_or(expanded_removal_array, 1 - clean_x.to(torch.uint8))

model_AT_rFGSM.eval()
step_length = 0.001

x_next = border_point.clone().detach()
x_var = x_next.clone().detach().requires_grad_(True)
y_model = model_AT_rFGSM(x_var)
mal = y_model[:, 1]
ben = y_model[:, 0]
loss = (mal - ben)
print(loss)

# Compute gradient
grad_vars = torch.autograd.grad(loss.mean(), x_var)
gradients = -(grad_vars[0].data)

gradients = removal_array_updated * gradients


l2norm = torch.linalg.norm(gradients, dim=-1, keepdim=True)
f_norm = (gradients / (l2norm + 1e-20)).float()


perturbation = delta_norm - (delta_norm * f_norm)
#perturbation2 = loss * f_norm

x_next2 = torch.clamp(x_next + perturbation * step_length, min=0., max=1.)
y_model = model_AT_rFGSM(x_next2)
real = y_model[:, 1]
other = y_model[:, 0]
loss = (real - other)
print(loss)


perturbation2 = loss * f_norm


x_next = torch.clamp(x_next + perturbation * step_length + perturbation2* 0.001 , min=0., max=1.)
y_model = model_AT_rFGSM(x_next)
real = y_model[:, 1]
other = y_model[:, 0]
loss = (real - other)
print(loss)

In [ ]:
x_next[0][:100]

In [ ]:
(torch.abs(x_next - clean_x)).sum()

In [ ]:
model_AT_rFGSM(torch.round(x_next))

In [ ]:
(torch.abs(torch.round(x_next) - clean_x)).sum()

## Step 3 — Mimicry, in this notebook's form

# Mimicry

In [ ]:
for x_test, y_test in val_loader:
    x_test, y_test = x_test.to(torch.float32).to(device), y_test.to(device)
    mal_x_batch1, mal_y_batch1 = x_test[y_test.squeeze() == 1], y_test[y_test.squeeze() == 1]
    ben_x_batch1, ben_y_batch1 = x_test[y_test.squeeze() == 0], y_test[y_test.squeeze() == 0]
    print('mal_x_batch1 ',mal_x_batch1.shape)
    print('ben_x_batch1 ',ben_x_batch1.shape)
    break

for x_test, y_test in test_loader:
    x_test, y_test = x_test.to(torch.float32).to(device), y_test.to(device)
    mal_x_batch2, mal_y_batch2 = x_test[y_test.squeeze() == 1], y_test[y_test.squeeze() == 1]
    ben_x_batch2, ben_y_batch2 = x_test[y_test.squeeze() == 0], y_test[y_test.squeeze() == 0]
    print('mal_x_batch2 ',mal_x_batch2.shape)
    print('ben_x_batch2 ',ben_x_batch2.shape)
    break

for x_test, y_test in train_loader:
    x_test, y_test = x_test.to(torch.float32).to(device), y_test.to(device)
    mal_x_batch0, mal_y_batch0 = x_test[y_test.squeeze() == 1], y_test[y_test.squeeze() == 1]
    ben_x_batch0, ben_y_batch0 = x_test[y_test.squeeze() == 0], y_test[y_test.squeeze() == 0]
    print('mal_x_batch0 ',mal_x_batch0.shape)
    print('ben_x_batch0 ',ben_x_batch0.shape)
    break

ben_x_batch = torch.cat((ben_x_batch0, ben_x_batch1, ben_x_batch2), dim=0)
print('ben_x_batch ',ben_x_batch.shape)

In [ ]:
# Move the model and static tensors to the GPU
model = combined_model.to(device)
model.eval()

removal_array = removal_array.to(device)
ben_x_batch = ben_x_batch.to(device)

# Initialize counters and storage for valid distances and their corresponding inputs
total_samples = 0
valid_sample_count = 0
valid_distances_sum = 0.0  # Sum of all valid distances
valid_distances_list = []  # Store valid distances for sorting later

# Iterate through mal_x_batch2
for x in mal_x_batch2:
    clean_x = x.unsqueeze(dim=0).to(device)
    expanded_removal_array = removal_array.expand(clean_x.shape[0], -1)
    non_removal_array_updated = torch.bitwise_and(1 - expanded_removal_array.to(torch.uint8), clean_x.to(torch.uint8))

    with torch.no_grad():
        # Create a batch of inputs by applying bitwise_or with non_removal_array_updated and ben_x_batch
        inputs = torch.bitwise_or(non_removal_array_updated, ben_x_batch.to(torch.uint8)).to(torch.float32)

        # Calculate the L2 distances between the clean input and all inputs in the batch
        distances = torch.abs(clean_x - inputs).sum(dim=1)

        # Pass all inputs through the model in a batch
        outputs = model(inputs)

        # Mask to filter out samples where outputs.argmax(dim=1) == 1
        mask = outputs.argmax(dim=1) == 0
        valid_distances = distances[mask]

        # Increment the total sample counter
        total_samples += 1

        # Check if there are any valid samples left
        if valid_distances.numel() > 0:
            # Find the minimum distance among the valid samples
            min_distance = torch.min(valid_distances).item()

            # Update the valid sample count and accumulate the sum of valid distances
            valid_sample_count += 1
            valid_distances_sum += min_distance

            # Store the distance in the list for later analysis
            valid_distances_list.append(min_distance)

            print(f"Closest distance for this sample: {min_distance}")
        else:
            print("No valid samples found with outputs.argmax() == 0")

# Calculate the percentage of samples with valid distances
valid_percentage = (valid_sample_count / total_samples) * 100 if total_samples > 0 else 0
# Calculate the mean distance for valid samples
mean_distance = valid_distances_sum / valid_sample_count if valid_sample_count > 0 else 0

# Sort valid distances to find the first 100 with the lowest distance
valid_distances_list.sort()
first_100_mean = sum(valid_distances_list[:100]) / min(100, len(valid_distances_list)) if valid_distances_list else 0
first_150_mean = sum(valid_distances_list[:150]) / min(150, len(valid_distances_list)) if valid_distances_list else 0
first_200_mean = sum(valid_distances_list[:200]) / min(200, len(valid_distances_list)) if valid_distances_list else 0

# Print the results
print('-----------------------------------------------------------------------')
print(f"Nmuber of valid samples: {valid_sample_count}")
print(f"Percentage of valid samples: {valid_percentage}%")
print(f"Mean distance for all valid samples: {mean_distance}")
print(f"Mean distance of the first 100 valid samples with the lowest distance: {first_100_mean}")
print(f"Mean distance of the first 150 valid samples with the lowest distance: {first_150_mean}")
print(f"Mean distance of the first 200 valid samples with the lowest distance: {first_200_mean}")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Sort the list of valid distances to prepare for cumulative counting
valid_distances_list.sort()

# Generate the x-axis (distance values) and y-axis (cumulative counts)
x_values = valid_distances_list
y_values = list(range(1, len(valid_distances_list) + 1))

# Set a style using Seaborn
sns.set(style='whitegrid')

# Create the plot
plt.figure(figsize=(10, 6))
sns.lineplot(x=x_values, y=y_values, marker='o', color='b', linewidth=2, markersize=5)

# Add labels and title with a bit more formatting
plt.xlabel('Distance', fontsize=14, labelpad=10)
plt.ylabel('Number of Valid Samples (Cumulative)', fontsize=14, labelpad=10)
plt.title('Cumulative Count of Valid Samples vs. Distance', fontsize=16, pad=15)

# Improve the look of the grid
plt.grid(True, linestyle='--', linewidth=0.5)

# Show the plot
plt.tight_layout()
plt.show()

## Step 4 — PGD from the mimicry/boundary starts

# PGD

In [ ]:
import torch.optim.lr_scheduler as lr_scheduler
from torch.optim.lr_scheduler import CyclicLR
import math

def cyclical_lr(step, step_size_up, step_size_down, min_lr, max_lr):
    cycle_length = step_size_up + step_size_down
    cycle_position = step % cycle_length

    if cycle_position < step_size_up:
        lr = min_lr + (max_lr - min_lr) * (cycle_position / step_size_up)
    else:
        lr = max_lr - (max_lr - min_lr) * ((cycle_position - step_size_up) / step_size_down)

    return lr

def lr_cosine_half(t, T, lr_start, lr_end):
    return lr_start + (lr_end - lr_start) * (1 - np.cos(np.pi * t /(2*T)))

def compare(output):
    output = torch.argmax(output, dim=1)
    return output == 0

def PGD(x, y, model, removal_array, max_iterations, min_lr, max_lr, step_size_up=2000,step_size_down=2000, norm='l2', confidence=0.1, device=device):
    """
    Projected Gradient Descent (PGD) adversarial attack (loss based on goal's class, which we have to minimize the loss).
    :param y: Ground truth labels
    :param x: Feature vector
    :param model: Neural network model
    :param k: Number of steps
    :param step_length: Step size for each iteration
    :param norm: Norm used for perturbation ('linf' or 'l2')
    :param initial_rounding_threshold: Threshold parameter for rounding the initial x_next
    :param round_threshold: Threshold parameter for rounding
    :param random: Flag to generate random thresholds
    :param is_report_loss_diff: Flag to report loss difference
    :param is_sample: Flag to sample randomly from the feasible area
    :return: The adversarial version of x (tensor)
    """
    model.eval()
    model.to(device)

    # Initialize starting point
    x_next = x.clone().to(device)
    #x_next = get_x0(x_next, initial_rounding_threshold, is_sample)

    # Expand insertion_array and removal_array to match the batch size
    expanded_removal_array = removal_array.expand(x.shape[0], -1)

    # Update insertion and removal arrays based on input x
    removal_array_updated = torch.bitwise_or(expanded_removal_array, 1 - x.to(torch.uint8)).to(device)

    batch_size = x.shape[0]
    success_mask = torch.zeros(batch_size, dtype=torch.bool, device=device)
    o_bestattack = x.clone().detach()
    o_bestl2 = torch.full((batch_size,), 1e10, device=device)

    for t in range(max_iterations):
        #print('*********** ',t)
        # Forward pass
        x_var = x_next.clone().detach().requires_grad_(True)

        y_model = model(x_var)

        real = y_model[:, 0]
        other = y_model[:, 1]
        loss = torch.clamp((other - real + confidence), min=0.0)
        #loss = (other - real)

        with torch.no_grad():
            perturbation = torch.where((x==1.), (x - x_next), (x_next - x))
            l2dist = torch.sum(perturbation, dim=1)

            success_mask_inloop = compare(y_model)
            success_mask  = torch.logical_or(success_mask_inloop, success_mask)

            update_mask2 = (l2dist < o_bestl2) & success_mask_inloop
            o_bestl2 = torch.where(update_mask2, l2dist, o_bestl2)
            o_bestattack = torch.where(update_mask2.unsqueeze(-1), x_var, o_bestattack)

        # Compute gradient
        grad_vars = torch.autograd.grad(loss.mean(), x_var)
        gradients = -(grad_vars[0].data)
        #print('torch.abs(gradients).sum() : ',torch.abs(gradients).sum(dim=-1))

        #grad4insertion = (gradients >= 0) *(x_var < 1.)* insertion_array_updated * gradients
        grad4removal = (gradients < 0) * (x_var > 0.) * removal_array_updated * gradients

        grad4insertion = (gradients >= 0) *(x_var < 1.)* gradients
        #grad4removal = (gradients < 0) * (x_var > 0.) * gradients

        gradients = grad4removal + grad4insertion
        #print(torch.abs(gradients).sum())

        # Norm
        if norm == 'linf':
            perturbation = torch.sign(gradients).float()

        elif norm == 'free':
            perturbation = gradients

        elif norm == 'max':
            max_values, _ = torch.max(torch.abs(gradients), dim=-1, keepdim=True)
            perturbation = (gradients / (max_values+ 1e-20))

        elif norm == 'l2':
            l2norm = torch.linalg.norm(gradients, dim=-1, keepdim=True)
            perturbation = (gradients / (l2norm + 1e-20)).float()

        elif norm == 'l1':
            # consider just features of a sample which are not updated yet(because our update is 0to1 or 1to0 not stepwise)
            un_mod = torch.abs(x - x_var) <= 1e-6
            gradients = gradients * un_mod
            max_grad = torch.topk(torch.abs(gradients).view(gradients.size(0), -1), 1, dim=-1)[0]
            #print('max_grad ',max_grad)
            perturbation = (torch.abs(gradients) >= max_grad).float() * torch.sign(gradients).float()
            done = get_done(x_next, y, model)
            if torch.all(done):
                break
            perturbation[done] = 0.

        else:
            raise ValueError("Expect 'l1' or 'l2' or 'linf' norm.")

        # Update x_next
        #step_length = cyclical_lr(t, step_size_up, step_size_down, min_lr, max_lr)
        step_length = lr_cosine_half(t, max_iterations, min_lr, max_lr)
        x_next = torch.clamp(x_next + perturbation * step_length, min=0., max=1.)

        if t == 0 or (t + 1) % (max_iterations // 20) == 0:
          print(f"Iteration {t + 1:5}:\t Current LR = {step_length:.4f}\t success = {success_mask.sum().item()}\t Dist = {o_bestl2[success_mask].mean().item():.2f}\t Loss = {loss.mean().item():.2f}")

    return o_bestattack

In [ ]:
#loss = (other - real)

attack_params =  {'confidence':0.1, 'removal_array':removal_array,'max_iterations': 30000,
                  'max_lr': 10, 'min_lr': 1e-4}
adv = adv_predict(test_loader, combined_model, PGD, device, **attack_params)

In [ ]:
for x_test, y_test in test_loader:
    x_test, y_test = x_test.to(torch.float32).to(device), y_test.to(device)
    mal_x_batch, mal_y_batch = x_test[y_test.squeeze() == 1], y_test[y_test.squeeze() == 1]
    break

In [ ]:
results_non_rounded = evaluate_non_rounded_adversarial_examples(mal_x_batch, mal_y_batch, adv, combined_model)
print(' ')
results_rounded = evaluate_rounded_adversarial_examples(mal_x_batch, mal_y_batch, adv, combined_model, removal_array)

In [ ]:
adv_rounded = results_rounded["adv_rounded"]
results_fixed_elements = ensure_fixed_elements_unchanged(mal_x_batch, adv, adv_rounded, removal_array)

In [ ]:
outputs = combined_model(adv_rounded)
y_pred = outputs.argmax(dim=1)

total_difference = torch.abs(adv_rounded[y_pred == 0] - mal_x_batch[y_pred == 0]).sum(1)
# Step 2: Sort the differences in ascending order
sorted_differences, _ = torch.sort(total_difference)

# Generate the x-axis (distance values) and y-axis (cumulative counts)
x_values = sorted_differences.cpu()
y_values = list(range(1, len(sorted_differences) + 1))

# Set a style using Seaborn
sns.set(style='whitegrid')

# Create the plot
plt.figure(figsize=(10, 6))
sns.lineplot(x=x_values, y=y_values, marker='o', color='b', linewidth=2, markersize=5)

# Add labels and title with a bit more formatting
plt.xlabel('Distance', fontsize=14, labelpad=10)
plt.ylabel('Number of Valid Samples (Cumulative)', fontsize=14, labelpad=10)
plt.title('Cumulative Count of Valid Samples vs. Distance', fontsize=16, pad=15)

# Improve the look of the grid
plt.grid(True, linestyle='--', linewidth=0.5)

# Show the plot
plt.tight_layout()
plt.show()

In [ ]:
#loss = torch.clamp((other - real + confidence), min=0.0)

attack_params =  {'confidence':0.1, 'removal_array':removal_array,'max_iterations': 30000,
                  'max_lr': 10, 'min_lr': 1e-4}
adv = adv_predict(test_loader, combined_model, PGD, device, **attack_params)

# keeping only non removal features

In [ ]:
import torch.optim.lr_scheduler as lr_scheduler
from torch.optim.lr_scheduler import CyclicLR
import math

def cyclical_lr(step, step_size_up, step_size_down, min_lr, max_lr):
    cycle_length = step_size_up + step_size_down
    cycle_position = step % cycle_length

    if cycle_position < step_size_up:
        lr = min_lr + (max_lr - min_lr) * (cycle_position / step_size_up)
    else:
        lr = max_lr - (max_lr - min_lr) * ((cycle_position - step_size_up) / step_size_down)

    return lr

def lr_cosine_half(t, T, lr_start, lr_end):
    return lr_start + (lr_end - lr_start) * (1 - np.cos(np.pi * t /(2*T)))

def compare(output):
    output = torch.argmax(output, dim=1)
    return output == 0

def PGD(x, y, model, removal_array, max_iterations, min_lr, max_lr, step_size_up=2000,step_size_down=2000, norm='l2', confidence=0.1, device=device):
    """
    Projected Gradient Descent (PGD) adversarial attack (loss based on goal's class, which we have to minimize the loss).
    :param y: Ground truth labels
    :param x: Feature vector
    :param model: Neural network model
    :param k: Number of steps
    :param step_length: Step size for each iteration
    :param norm: Norm used for perturbation ('linf' or 'l2')
    :param initial_rounding_threshold: Threshold parameter for rounding the initial x_next
    :param round_threshold: Threshold parameter for rounding
    :param random: Flag to generate random thresholds
    :param is_report_loss_diff: Flag to report loss difference
    :param is_sample: Flag to sample randomly from the feasible area
    :return: The adversarial version of x (tensor)
    """
    model.eval()
    model.to(device)

    # Initialize starting point
    #x_next = x.clone().to(device)
    #x_next = get_x0(x_next, initial_rounding_threshold, is_sample)

    # Expand insertion_array and removal_array to match the batch size
    expanded_removal_array = removal_array.expand(x.shape[0], -1)

    # Update insertion and removal arrays based on input x
    removal_array_updated = torch.bitwise_or(expanded_removal_array, 1 - x.to(torch.uint8)).to(device)

    # we keep only non removal features
    x_next = torch.bitwise_and(1 - expanded_removal_array.to(torch.uint8), x.to(torch.uint8)).to(torch.float32).to(device)

    batch_size = x.shape[0]
    success_mask = torch.zeros(batch_size, dtype=torch.bool, device=device)
    o_bestattack = x.clone().detach()
    o_bestl2 = torch.full((batch_size,), 1e10, device=device)

    for t in range(max_iterations):
        #print('*********** ',t)
        # Forward pass
        x_var = x_next.clone().detach().requires_grad_(True)

        y_model = model(x_var)

        real = y_model[:, 0]
        other = y_model[:, 1]
        #loss = torch.clamp((other - real + confidence), min=0.0)
        loss = (other - real)

        with torch.no_grad():
            perturbation = torch.where((x==1.), (x - x_next), (x_next - x))
            l2dist = torch.sum(perturbation, dim=1)

            success_mask_inloop = compare(y_model)
            success_mask  = torch.logical_or(success_mask_inloop, success_mask)

            update_mask2 = (l2dist < o_bestl2) & success_mask_inloop
            o_bestl2 = torch.where(update_mask2, l2dist, o_bestl2)
            o_bestattack = torch.where(update_mask2.unsqueeze(-1), x_var, o_bestattack)

        # Compute gradient
        grad_vars = torch.autograd.grad(loss.mean(), x_var)
        gradients = -(grad_vars[0].data)
        #print('torch.abs(gradients).sum() : ',torch.abs(gradients).sum(dim=-1))

        #grad4insertion = (gradients >= 0) *(x_var < 1.)* insertion_array_updated * gradients
        grad4removal = (gradients < 0) * (x_var > 0.) * removal_array_updated * gradients

        grad4insertion = (gradients >= 0) *(x_var < 1.)* gradients
        #grad4removal = (gradients < 0) * (x_var > 0.) * gradients

        gradients = grad4removal + grad4insertion
        #print(torch.abs(gradients).sum())

        # Norm
        if norm == 'linf':
            perturbation = torch.sign(gradients).float()

        elif norm == 'free':
            perturbation = gradients

        elif norm == 'max':
            max_values, _ = torch.max(torch.abs(gradients), dim=-1, keepdim=True)
            perturbation = (gradients / (max_values+ 1e-20))

        elif norm == 'l2':
            l2norm = torch.linalg.norm(gradients, dim=-1, keepdim=True)
            perturbation = (gradients / (l2norm + 1e-20)).float()

        elif norm == 'l1':
            # consider just features of a sample which are not updated yet(because our update is 0to1 or 1to0 not stepwise)
            un_mod = torch.abs(x - x_var) <= 1e-6
            gradients = gradients * un_mod
            max_grad = torch.topk(torch.abs(gradients).view(gradients.size(0), -1), 1, dim=-1)[0]
            #print('max_grad ',max_grad)
            perturbation = (torch.abs(gradients) >= max_grad).float() * torch.sign(gradients).float()
            done = get_done(x_next, y, model)
            if torch.all(done):
                break
            perturbation[done] = 0.

        else:
            raise ValueError("Expect 'l1' or 'l2' or 'linf' norm.")

        # Update x_next
        #step_length = cyclical_lr(t, step_size_up, step_size_down, min_lr, max_lr)
        step_length = lr_cosine_half(t, max_iterations, min_lr, max_lr)
        x_next = torch.clamp(x_next + perturbation * step_length, min=0., max=1.)

        if t == 0 or (t + 1) % (max_iterations // 20) == 0:
          print(f"Iteration {t + 1:5}:\t Current LR = {step_length:.4f}\t success = {success_mask.sum().item()}\t Dist = {o_bestl2[success_mask].mean().item():.2f}\t Loss = {loss.mean().item():.2f}")

    return o_bestattack

## Step 5 — CW from x, and from the non-removal features of x

# CW

## Start from x

In [ ]:
import torch
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from torch.optim.lr_scheduler import OneCycleLR
from torch.optim.lr_scheduler import LambdaLR

def compare(output):
    output = torch.argmax(output, dim=1)
    return output == 0

def sin_func_torch(input_tensor):
    """
    A sine-based transformation applied to the input tensor.
    The function applies a sine transformation and rescales the output to [0, 1].
    """
    return (torch.sin((input_tensor - 0.5) * torch.pi) + 1) / 2.0


# Define the learning rate schedules (same as earlier)
def lr_linear(t, T, lr_start=1e-3, lr_end=0.9):
    return lr_start + t * (lr_end - lr_start) / T

def lr_exponential(t, T, lr_start=1e-3, lr_end=0.9):
    return lr_start * (lr_end / lr_start) ** (t / T)

def lr_cosine(t, T, lr_start=1e-3, lr_end=0.9):
    return lr_end + (lr_start - lr_end) * 0.5 * (1 + torch.cos(torch.tensor(np.pi * t / T)))

def lr_average(t, T, lr_start=1e-3, lr_end=0.9):
    return 0.5 * (lr_linear(t, T, lr_start, lr_end) + lr_exponential(t, T, lr_start, lr_end))

def lr_cosine_half(t, T, lr_start, lr_end):
    return lr_start + (lr_end - lr_start) * (1 - np.cos(np.pi * t /(2*T)))



def CW(model, imgs, y, removal_array, device='cuda', initial_const = 1., binary_search_steps=8, cons_increase_factor=10., LR_function=lr_exponential,
                 max_lr=2., min_lr=1e-4,update_step=1000, first_phase_iters=20000,second_phase_iters = 10000 , confidence=0.1, eps=1e-4):
    """
    Perform a Learning Rate Range Test for adversarial attack optimization using the Adam optimizer.

    - model: The target model to attack.
    - imgs: The original images to be attacked.
    - y: The true labels of the images.
    - removal_array: A mask array specifying which features are fixed.
    - device: The device on which to perform computations.
    - max_lr: Maximum learning rate to end with.
    - num_iter: Number of iterations to perform.
    - confidence: Confidence parameter for adversarial loss computation.
    """

    def grad_hook(grad):
        return grad * non_fixed_features_mask

    # Preparing tensors
    imgs_tensor_original = imgs.clone().detach().to(dtype=torch.float32, device=device)
    imgs_tensor = torch.where(imgs.clone().detach().to(torch.uint8), torch.pi * 0.5, -torch.pi * 0.5).to(device)
    labs_tensor = torch.zeros_like(y.squeeze(), dtype=torch.int64, device=device)

    batch_size = imgs.shape[0]

    # Create masks for 0s and 1s
    zero_mask = (imgs == 0)  # Mask where values are 0
    one_mask = (imgs == 1)   # Mask where values are 1

    # Add eps to 0s and subtract eps from 1s
    imgs_tensor[zero_mask] += eps
    imgs_tensor[one_mask] -= eps

    lower_bound = torch.zeros(batch_size, dtype=torch.float32, device=device)
    CONST = torch.full((batch_size,), initial_const, dtype=torch.float32, device=device)
    upper_bound = torch.full((batch_size,), 1e10, dtype=torch.float32, device=device)

    # Expand removal array to match the batch size
    expanded_removal_array = removal_array.expand(imgs.shape[0], -1)
    non_fixed_features_mask = torch.bitwise_or(expanded_removal_array, 1 - imgs.to(torch.uint8))


    # Initialize tracking variables
    o_bestl2 = torch.full((batch_size,), 1e10, device=device)
    o_bestattack = imgs.clone().detach()

    o_best_loss1 = torch.full((batch_size,), float('inf'), device=device)
    o_best_unsuccessfull_attack = imgs.clone().detach()
    o_success_mask = torch.zeros(batch_size, dtype=torch.bool, device=device)

    model.eval()

    # Total number of iterations
    total_iters = first_phase_iters + second_phase_iters

    for outer_step in range(binary_search_steps):
        print('------------------------------------------------------------------------------------------------------------------------------------')
        print(CONST)

        modifier = torch.zeros_like(imgs_tensor, requires_grad=True).to(device)
        optimizer = torch.optim.Adam([modifier], lr=min_lr)
        modifier.register_hook(grad_hook)

        bestl2 = torch.full((batch_size,), 1e10, device=device)
        bestattack = imgs.clone().detach()
        best_modifier = torch.zeros_like(modifier).detach()  # Best modifier
        best_unsuccessfull_modifier = torch.zeros_like(modifier).detach()

        best_loss1 = torch.full((batch_size,), float('inf'), device=device)
        best_unsuccessfull_attack = imgs.clone().detach()
        success_mask = torch.zeros(batch_size, dtype=torch.bool, device=device)

        for iteration in range(total_iters):

            # Phase 2: Fixed learning rate of 1e-3 (iterations 15,000 to 24,999)
            if iteration == first_phase_iters:
                print("************ second phase *************")
                output = model(bestattack)
                final_success = compare(output)
                final_output = torch.where(final_success.unsqueeze(-1), bestattack, best_unsuccessfull_attack)
                final_modifier = torch.where(final_success.unsqueeze(-1), best_modifier, best_unsuccessfull_modifier)

                modifier = torch.tensor(final_modifier.clone().detach(), requires_grad=True).to(device)
                optimizer = torch.optim.Adam([modifier], lr=1e-3)  # Use lr_min for 1000 iterations
                modifier.register_hook(grad_hook)

            if iteration < first_phase_iters:
                if iteration == 0 or (iteration + 1) % update_step == 0:
                    # Select your preferred learning rate schedule
                    current_lr = LR_function(iteration, first_phase_iters, min_lr, max_lr)  # You can switch to lr_linear, lr_exponential, or lr_cosine

                    # Update the optimizer's learning rate
                    for param_group in optimizer.param_groups:
                        param_group['lr'] = current_lr

            optimizer.zero_grad()

            # Attack Step: generate perturbed image
            modified_input = imgs_tensor + modifier
            newimg = torch.sin(modified_input) * 0.5 + 0.5
            output = model(newimg)

            l2dist = torch.sum((newimg - imgs_tensor_original) ** 2, dim=1)

            real = output[:, 0]
            other = output[:, 1]

            loss1 = torch.clamp((other - real + confidence), min=0.0)
            loss = (CONST * loss1) + l2dist

            total_loss = torch.sum(loss)
            total_loss.backward()
            optimizer.step()

            with torch.no_grad():
                success_mask_inloop = compare(output)
                success_mask = torch.logical_or(success_mask_inloop, success_mask)
                o_success_mask  = torch.logical_or(o_success_mask, success_mask_inloop)

                update_mask = (l2dist < bestl2) & success_mask_inloop
                bestl2 = torch.where(update_mask, l2dist, bestl2)
                bestattack = torch.where(update_mask.unsqueeze(-1), newimg, bestattack)
                best_modifier = torch.where(update_mask.unsqueeze(-1), modifier.detach(), best_modifier)

                improved_mask = (loss1 < best_loss1)
                best_loss1 = torch.where(improved_mask, loss1, o_best_loss1)
                best_unsuccessfull_attack = torch.where(improved_mask.unsqueeze(-1), newimg, best_unsuccessfull_attack)
                best_unsuccessfull_modifier = torch.where(improved_mask.unsqueeze(-1), modifier.detach(), best_unsuccessfull_modifier)

                update_mask2 = (l2dist < o_bestl2) & success_mask_inloop
                o_bestl2 = torch.where(update_mask2, l2dist, o_bestl2)
                o_bestattack = torch.where(update_mask2.unsqueeze(-1), newimg, o_bestattack)

                improved_mask2 = (loss1 < o_best_loss1)
                o_best_loss1 = torch.where(improved_mask2, loss1, o_best_loss1)
                o_best_unsuccessfull_attack = torch.where(improved_mask2.unsqueeze(-1), newimg, o_best_unsuccessfull_attack)


                if iteration == 0 or (iteration + 1) % 500 == 0:
                    current_lr = optimizer.param_groups[0]['lr']
                    current_momentum, current_momentum2 = optimizer.param_groups[0]['betas']
                    print(f"Iteration {iteration + 1:5}:\t Current LR = {current_lr:.4f} \t Current momentum = {current_momentum:.2f} \t Current success = {success_mask_inloop.sum().item()} \t success = {success_mask.sum().item()} \t Loss = {total_loss.item():.4f} \t l2dist = {torch.mean(bestl2[success_mask]).item():.4f}")

                    print(f"Best results until now: \t success = {o_success_mask.sum().item()} \t l2dist = {torch.mean(o_bestl2[o_success_mask]).item():.4f}")

        failure_mask = ~success_mask_inloop
        upper_bound[success_mask_inloop] = torch.minimum(upper_bound[success_mask_inloop], CONST[success_mask_inloop])
        lower_bound[failure_mask] = torch.maximum(lower_bound[failure_mask], CONST[failure_mask])

        upper_limit_mask = upper_bound < 1e9
        CONST[upper_limit_mask] = (lower_bound[upper_limit_mask] + upper_bound[upper_limit_mask]) / 2
        CONST[failure_mask & ~upper_limit_mask] *= cons_increase_factor



    print('')
    output = model(o_bestattack)
    final_success = compare(output)
    print(f"FINAL RESULT :\t #success = {final_success.sum().item()} \t total Loss = {total_loss.item():.4f}\t success Loss = {loss[final_success].sum().item():.4f} \t success l2dist = {torch.mean(o_bestl2[final_success]).item():.4f}")
    print('')

    final_output = torch.where(final_success.unsqueeze(-1), o_bestattack, o_best_unsuccessfull_attack)
    results_rounded = evaluate_rounded_adversarial_examples(imgs, y, final_output, combined_model, removal_array)
    print('')

In [ ]:
CW(combined_model, mal_x_batch, mal_y_batch, removal_array, device='cuda', initial_const = 1., binary_search_steps=8, cons_increase_factor=10., LR_function=lr_cosine_half,
                 max_lr=2.5, min_lr=1e-3, update_step=500, first_phase_iters=20000,second_phase_iters = 5000 , confidence=0.1, eps=1e-4)

## Start from non removal featurs of x

In [ ]:
import torch
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from torch.optim.lr_scheduler import OneCycleLR
from torch.optim.lr_scheduler import LambdaLR

def compare(output):
    output = torch.argmax(output, dim=1)
    return output == 0

def sin_func_torch(input_tensor):
    """
    A sine-based transformation applied to the input tensor.
    The function applies a sine transformation and rescales the output to [0, 1].
    """
    return (torch.sin((input_tensor - 0.5) * torch.pi) + 1) / 2.0


# Define the learning rate schedules (same as earlier)
def lr_linear(t, T, lr_start=1e-3, lr_end=0.9):
    return lr_start + t * (lr_end - lr_start) / T

def lr_exponential(t, T, lr_start=1e-3, lr_end=0.9):
    return lr_start * (lr_end / lr_start) ** (t / T)

def lr_cosine(t, T, lr_start=1e-3, lr_end=0.9):
    return lr_end + (lr_start - lr_end) * 0.5 * (1 + torch.cos(torch.tensor(np.pi * t / T)))

def lr_average(t, T, lr_start=1e-3, lr_end=0.9):
    return 0.5 * (lr_linear(t, T, lr_start, lr_end) + lr_exponential(t, T, lr_start, lr_end))

def lr_cosine_half(t, T, lr_start, lr_end):
    return lr_start + (lr_end - lr_start) * (1 - np.cos(np.pi * t /(2*T)))



def CW(model, x, y, removal_array, device='cuda', initial_const = 1., binary_search_steps=8, cons_increase_factor=10., LR_function=lr_exponential,
                 max_lr=2., min_lr=1e-4,update_step=1000, first_phase_iters=20000,second_phase_iters = 10000 , confidence=0.1, eps=1e-4):
    """
    Perform a Learning Rate Range Test for adversarial attack optimization using the Adam optimizer.

    - model: The target model to attack.
    - imgs: The original images to be attacked.
    - y: The true labels of the images.
    - removal_array: A mask array specifying which features are fixed.
    - device: The device on which to perform computations.
    - max_lr: Maximum learning rate to end with.
    - num_iter: Number of iterations to perform.
    - confidence: Confidence parameter for adversarial loss computation.
    """

    def grad_hook(grad):
        return grad * non_fixed_features_mask

    # Expand removal array to match the batch size
    expanded_removal_array = removal_array.expand(x.shape[0], -1)
    non_fixed_features_mask = torch.bitwise_or(expanded_removal_array, 1 - x.to(torch.uint8))

    # we keep only non removal features
    imgs = torch.bitwise_and(1 - expanded_removal_array.to(torch.uint8), x.to(torch.uint8)).to(torch.float32).to(device)

    # Preparing tensors
    imgs_tensor_original = x.clone().detach().to(dtype=torch.float32, device=device)
    imgs_tensor = torch.where(imgs.clone().detach().to(torch.uint8), torch.pi * 0.5, -torch.pi * 0.5).to(device)
    labs_tensor = torch.zeros_like(y.squeeze(), dtype=torch.int64, device=device)

    batch_size = imgs.shape[0]

    # Create masks for 0s and 1s
    zero_mask = (imgs == 0)  # Mask where values are 0
    one_mask = (imgs == 1)   # Mask where values are 1

    # Add eps to 0s and subtract eps from 1s
    imgs_tensor[zero_mask] += eps
    #imgs_tensor[one_mask] -= eps

    lower_bound = torch.zeros(batch_size, dtype=torch.float32, device=device)
    CONST = torch.full((batch_size,), initial_const, dtype=torch.float32, device=device)
    upper_bound = torch.full((batch_size,), 1e10, dtype=torch.float32, device=device)


    # Initialize tracking variables
    o_bestl2 = torch.full((batch_size,), 1e10, device=device)
    o_bestattack = imgs.clone().detach()

    o_best_loss1 = torch.full((batch_size,), float('inf'), device=device)
    o_best_unsuccessfull_attack = imgs.clone().detach()
    o_success_mask = torch.zeros(batch_size, dtype=torch.bool, device=device)

    model.eval()

    # Total number of iterations
    total_iters = first_phase_iters + second_phase_iters

    for outer_step in range(binary_search_steps):
        print('------------------------------------------------------------------------------------------------------------------------------------')
        print(CONST)

        modifier = torch.zeros_like(imgs_tensor, requires_grad=True).to(device)
        optimizer = torch.optim.Adam([modifier], lr=min_lr)
        modifier.register_hook(grad_hook)

        bestl2 = torch.full((batch_size,), 1e10, device=device)
        bestattack = imgs.clone().detach()
        best_modifier = torch.zeros_like(modifier).detach()  # Best modifier
        best_unsuccessfull_modifier = torch.zeros_like(modifier).detach()

        best_loss1 = torch.full((batch_size,), float('inf'), device=device)
        best_unsuccessfull_attack = imgs.clone().detach()
        success_mask = torch.zeros(batch_size, dtype=torch.bool, device=device)

        for iteration in range(total_iters):

            # Phase 2: Fixed learning rate of 1e-3 (iterations 15,000 to 24,999)
            if iteration == first_phase_iters:
                print("************ second phase *************")
                output = model(o_bestattack)
                final_success = compare(output)
                final_output = torch.where(final_success.unsqueeze(-1), bestattack, best_unsuccessfull_attack)
                final_modifier = torch.where(final_success.unsqueeze(-1), best_modifier, best_unsuccessfull_modifier)

                modifier = torch.tensor(final_modifier.clone().detach(), requires_grad=True).to(device)
                optimizer = torch.optim.Adam([modifier], lr=1e-3)  # Use lr_min for 1000 iterations
                modifier.register_hook(grad_hook)

            if iteration < first_phase_iters:
                if iteration == 0 or (iteration + 1) % update_step == 0:
                    # Select your preferred learning rate schedule
                    current_lr = LR_function(iteration, first_phase_iters, min_lr, max_lr)  # You can switch to lr_linear, lr_exponential, or lr_cosine

                    # Update the optimizer's learning rate
                    for param_group in optimizer.param_groups:
                        param_group['lr'] = current_lr

            optimizer.zero_grad()

            # Attack Step: generate perturbed image
            modified_input = imgs_tensor + modifier
            newimg = torch.sin(modified_input) * 0.5 + 0.5
            output = model(newimg)

            l2dist = torch.sum((newimg - imgs_tensor_original) ** 2, dim=1)

            real = output[:, 0]
            other = output[:, 1]

            loss1 = torch.clamp((other - real + confidence), min=0.0)
            loss = (CONST * loss1) + l2dist

            total_loss = torch.sum(loss)
            total_loss.backward()
            optimizer.step()

            with torch.no_grad():
                success_mask_inloop = compare(output)
                success_mask = torch.logical_or(success_mask_inloop, success_mask)
                o_success_mask  = torch.logical_or(o_success_mask, success_mask_inloop)

                update_mask = (l2dist < bestl2) & success_mask_inloop
                bestl2 = torch.where(update_mask, l2dist, bestl2)
                bestattack = torch.where(update_mask.unsqueeze(-1), newimg, bestattack)
                best_modifier = torch.where(update_mask.unsqueeze(-1), modifier.detach(), best_modifier)

                improved_mask = (loss1 < best_loss1)
                best_loss1 = torch.where(improved_mask, loss1, o_best_loss1)
                best_unsuccessfull_attack = torch.where(improved_mask.unsqueeze(-1), newimg, best_unsuccessfull_attack)
                best_unsuccessfull_modifier = torch.where(improved_mask.unsqueeze(-1), modifier.detach(), best_unsuccessfull_modifier)

                update_mask2 = (l2dist < o_bestl2) & success_mask_inloop
                o_bestl2 = torch.where(update_mask2, l2dist, o_bestl2)
                o_bestattack = torch.where(update_mask2.unsqueeze(-1), newimg, o_bestattack)

                improved_mask2 = (loss1 < o_best_loss1)
                o_best_loss1 = torch.where(improved_mask2, loss1, o_best_loss1)
                o_best_unsuccessfull_attack = torch.where(improved_mask2.unsqueeze(-1), newimg, o_best_unsuccessfull_attack)


                if iteration == 0 or (iteration + 1) % 500 == 0:
                    current_lr = optimizer.param_groups[0]['lr']
                    current_momentum, current_momentum2 = optimizer.param_groups[0]['betas']
                    print(f"Iteration {iteration + 1:5}:\t Current LR = {current_lr:.4f} \t Current momentum = {current_momentum:.2f} \t Current success = {success_mask_inloop.sum().item()} \t success = {success_mask.sum().item()} \t Loss = {total_loss.item():.4f} \t l2dist = {torch.mean(bestl2[success_mask]).item():.4f}")

                    print(f"Best results until now: \t success = {o_success_mask.sum().item()} \t l2dist = {torch.mean(o_bestl2[o_success_mask]).item():.4f}")

        failure_mask = ~success_mask
        upper_bound[success_mask] = torch.minimum(upper_bound[success_mask], CONST[success_mask])
        lower_bound[failure_mask] = torch.maximum(lower_bound[failure_mask], CONST[failure_mask])

        upper_limit_mask = upper_bound < 1e9
        CONST[upper_limit_mask] = (lower_bound[upper_limit_mask] + upper_bound[upper_limit_mask]) / 2
        CONST[failure_mask & ~upper_limit_mask] *= cons_increase_factor



    print('')
    output = model(o_bestattack)
    final_success = compare(output)
    print(f"FINAL RESULT :\t #success = {final_success.sum().item()} \t total Loss = {total_loss.item():.4f}\t success Loss = {loss[final_success].sum().item():.4f} \t success l2dist = {torch.mean(o_bestl2[final_success]).item():.4f}")
    print('')

    final_output = torch.where(final_success.unsqueeze(-1), o_bestattack, o_best_unsuccessfull_attack)
    results_rounded = evaluate_rounded_adversarial_examples(imgs, y, final_output, combined_model, removal_array)
    print('')

    return final_output, results_rounded["adv_rounded"]

In [ ]:
adv, adv_rounded = CW(combined_model, mal_x_batch, mal_y_batch, removal_array, device='cuda', initial_const = 1., binary_search_steps=8, cons_increase_factor=10., LR_function=lr_cosine_half,
                 max_lr=2.5, min_lr=1e-3, update_step=500, first_phase_iters=20000,second_phase_iters = 5000 , confidence=0.1, eps=1e-4)

In [ ]:
results_fixed_elements = ensure_fixed_elements_unchanged(mal_x_batch, adv, adv_rounded, removal_array)

## Step 6 — Mimicry + CW: the combined attack's functions

# CW : With Using mimicry results

success samples :

*   a1 : mimicry
*   a2 : borders

unsuccess samples :

*   b1 : original
*   b2 : non_removal fetures



## Norm L2

### Functions

In [ ]:
# Extract batches from the validation set
for x_test, y_test in val_loader:
    x_test = x_test.to(torch.float32).to(device)
    y_test = y_test.to(device)

    # Split into malicious and benign batches based on labels
    mal_x_batch1 = x_test[y_test.squeeze() == 1]
    ben_x_batch1 = x_test[y_test.squeeze() == 0]

    mal_y_batch1 = y_test[y_test.squeeze() == 1]
    ben_y_batch1 = y_test[y_test.squeeze() == 0]

    print(f'Malicious batch (validation): {mal_x_batch1.shape}')
    print(f'Benign batch (validation): {ben_x_batch1.shape}')
    break

# Extract batches from the test set
for x_test, y_test in test_loader:
    x_test = x_test.to(torch.float32).to(device)
    y_test = y_test.to(device)

    # Split into malicious and benign batches based on labels
    mal_x_batch2 = x_test[y_test.squeeze() == 1]
    ben_x_batch2 = x_test[y_test.squeeze() == 0]

    mal_y_batch2 = y_test[y_test.squeeze() == 1]
    ben_y_batch2 = y_test[y_test.squeeze() == 0]

    print(f'Malicious batch (test): {mal_x_batch2.shape}')
    print(f'Benign batch (test): {ben_x_batch2.shape}')
    break

# Extract batches from the training set
for x_test, y_test in train_loader:
    x_test = x_test.to(torch.float32).to(device)
    y_test = y_test.to(device)

    # Split into malicious and benign batches based on labels
    mal_x_batch0 = x_test[y_test.squeeze() == 1]
    ben_x_batch0 = x_test[y_test.squeeze() == 0]

    mal_y_batch0 = y_test[y_test.squeeze() == 1]
    ben_y_batch0 = y_test[y_test.squeeze() == 0]

    print(f'Malicious batch (training): {mal_x_batch0.shape}')
    print(f'Benign batch (training): {ben_x_batch0.shape}')
    break

# Concatenate the benign batches from validation and test sets
ben_x_batch = torch.cat((ben_x_batch1, ben_x_batch2), dim=0)
print(f'Combined benign batch: {ben_x_batch.shape}')

In [ ]:
def Mimicry(model, removal_array, ben_x_batch, mal_x_batch):
    # Move the model and static tensors to the GPU
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = model.to(device)
    model.eval()
    removal_array = removal_array.to(device)

    # Expand removal array to match the batch size
    expanded_removal_array = removal_array.expand(mal_x_batch.shape[0], -1).to(torch.uint8)
    non_fixed_features_mask = torch.bitwise_or(expanded_removal_array, 1 - mal_x_batch.to(torch.uint8))

    # Prepare tensors for storing results
    final_output_tensor1 = mal_x_batch.clone().to(device)
    final_output_tensor2 = torch.bitwise_and(
        1 - expanded_removal_array, mal_x_batch.to(torch.uint8)
    ).float().to(device)

    # Initialize counters and storage for valid distances
    total_samples = 0
    valid_sample_count = 0
    valid_distances_sum = 0.0
    valid_distances_list = []

    # Iterate through mal_x_batch with index tracking to store results
    for idx, x in enumerate(mal_x_batch):
        mal_sample = x.unsqueeze(0).to(device)
        ben_x_batch = ben_x_batch.to(device)

        non_removal_array_updated = torch.bitwise_and(
            1 - expanded_removal_array[0], mal_sample.to(torch.uint8)
        )

        # Create a batch of inputs by combining with benign samples
        ben_x_batch_new = torch.cat((mal_sample, ben_x_batch), dim=0)
        inputs = torch.bitwise_or(non_removal_array_updated, ben_x_batch_new.to(torch.uint8)).float()

        # Pass all inputs through the model in a batch
        with torch.no_grad():
            outputs = model(inputs)

        # Calculate L1 distances between the malicious input and the batch
        distances = torch.abs(mal_sample - inputs).sum(dim=1)

        # Filter out samples where the model predicts class 0 (mask == True)
        mask = outputs.argmax(dim=1) == 0
        valid_distances = distances[mask]
        valid_inputs = inputs[mask]

        # Increment the total sample counter
        total_samples += 1

        # Check for valid samples and update tensors
        if valid_distances.numel() > 0:
            min_distance, min_index = torch.min(valid_distances, dim=0)
            candidate_input = valid_inputs[min_index]

            # Store the valid candidate input into the final tensors
            final_output_tensor1[idx] = candidate_input
            final_output_tensor2[idx] = candidate_input

            # Update the valid sample count and distances
            valid_sample_count += 1
            valid_distances_sum += min_distance.item()
            valid_distances_list.append(min_distance.item())

        # Print status every 100 samples or at the start
        if idx == 0 or (idx + 1) % 100 == 0:
            current_mean_distance = valid_distances_sum / valid_sample_count if valid_sample_count > 0 else 0
            print(f"samples {idx + 1:5}:\t Current success = {valid_sample_count} \t l1dist = {current_mean_distance:.4f}")

    # Final print statement for the last batch
    current_mean_distance = valid_distances_sum / valid_sample_count if valid_sample_count > 0 else 0
    print(f"samples {idx + 1:5}:\t Current success = {valid_sample_count} \t l1dist = {current_mean_distance:.4f}")

    # Calculate the percentage of samples with valid distances and mean distance
    valid_percentage = (valid_sample_count / total_samples) * 100 if total_samples > 0 else 0
    mean_distance = valid_distances_sum / valid_sample_count if valid_sample_count > 0 else 0

    # Sort valid distances and calculate means for specific sets
    valid_distances_list.sort()
    first_100_mean = sum(valid_distances_list[:100]) / min(100, len(valid_distances_list)) if valid_distances_list else 0
    first_150_mean = sum(valid_distances_list[:150]) / min(150, len(valid_distances_list)) if valid_distances_list else 0
    first_200_mean = sum(valid_distances_list[:200]) / min(200, len(valid_distances_list)) if valid_distances_list else 0

    # Print the final results
    print('-----------------------------------------------------------------------')
    print(f"Number of valid samples: {valid_sample_count}")
    print(f"Percentage of valid samples: {valid_percentage:.2f}%")
    print(f"Mean distance for all valid samples: {mean_distance:.2f}")
    print(f"Mean distance of the first 100 valid samples with the lowest distance: {first_100_mean}")
    print(f"Mean distance of the first 150 valid samples with the lowest distance: {first_150_mean}")
    print(f"Mean distance of the first 200 valid samples with the lowest distance: {first_200_mean}")

    return final_output_tensor1, final_output_tensor2, valid_distances_list

In [ ]:
def Mimicry_group_one(model, removal_array, ben_x_batch, mal_x_batch):
    # Move the model and static tensors to the GPU
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = model.to(device)
    model.eval()
    removal_array = removal_array.to(device)

    # Expand removal array to match the batch size
    expanded_removal_array = removal_array.expand(mal_x_batch.shape[0], -1).to(torch.uint8)
    non_fixed_features_mask = torch.bitwise_or(expanded_removal_array, 1 - mal_x_batch.to(torch.uint8))

    # Prepare tensors for storing results
    final_output_tensor1 = mal_x_batch.clone().to(device)
    final_output_tensor2 = torch.bitwise_and(
        1 - expanded_removal_array, mal_x_batch.to(torch.uint8)
    ).float().to(device)

    # Initialize counters and storage for valid distances
    total_samples = 0
    valid_distances_sum = 0.0
    valid_distances_list = []
    mimic_success = 0
    valid_mimic_success = 0

    # Iterate through mal_x_batch with index tracking to store results
    for idx, x in enumerate(mal_x_batch):
        mal_sample = x.unsqueeze(0).to(device)
        ben_x_batch = ben_x_batch.to(device)

        non_removal_array_updated = torch.bitwise_and(
            1 - expanded_removal_array[0], mal_sample.to(torch.uint8)
        )

        # Create a batch of inputs by combining with benign samples
        #ben_x_batch_new = torch.cat((mal_sample, ben_x_batch), dim=0)
        ben_x_batch_new = ben_x_batch.clone()
        comparison = torch.bitwise_and(non_removal_array_updated, ben_x_batch_new.to(torch.uint8)).float()

        result = (non_removal_array_updated == comparison).all(dim=1)
        mimic_success += (result.sum() > 0).int()
        inputs = ben_x_batch_new[result].clone()


        # Calculate L1 distances between the malicious input and the batch
        distances = torch.abs(mal_sample - inputs).sum(dim=1)

        # Pass all inputs through the model in a batch
        with torch.no_grad():
            outputs = model(inputs)

        # Filter out samples where the model predicts class 0 (mask == True)
        mask = outputs.argmax(dim=1) == 0
        valid_distances = distances[mask]
        valid_inputs = inputs[mask]

        # Check for valid samples and update tensors
        if valid_distances.numel() > 0:
            min_distance, min_index = torch.min(valid_distances, dim=0)
            candidate_input_min = valid_inputs[min_index]

            with torch.no_grad():
                valid_mimic_success += (model(candidate_input_min.unsqueeze(0)).argmax(dim=1) == 0).int().item()

            # Store the valid candidate input into the final tensors
            final_output_tensor1[idx] = candidate_input_min.clone()
            final_output_tensor2[idx] = candidate_input_min.clone()

        # Print status every 100 samples or at the start
        if idx == 0 or (idx + 1) % 100 == 0:
            print(f"samples {idx + 1:5}:\t All mimic success = {mimic_success}\t Correct prediction of succesful mimic samples = {valid_mimic_success}")

    # Final print statement for the last batch
    print(f"samples {idx + 1:5}:\t All mimic success = {mimic_success}\t Correct prediction of succesful mimic samples = {valid_mimic_success}")


    return final_output_tensor1, final_output_tensor2, valid_distances_list

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

def plot_cumulative_distances(valid_distances_list):
    """
    Plots the cumulative count of valid samples against sorted distance values.

    Args:
        valid_distances_list (list): A list of valid distances.
    """
    # Ensure the list is sorted for cumulative counting
    valid_distances_list.sort()

    # Generate the x-axis (distance values) and y-axis (cumulative counts)
    x_values = np.array(valid_distances_list)
    y_values = np.arange(1, len(valid_distances_list) + 1)

    # Set Seaborn style for better visual appeal
    sns.set(style='whitegrid')

    # Create the plot with improved aesthetics
    plt.figure(figsize=(10, 6))
    sns.lineplot(x=x_values, y=y_values, marker='o', color='b', linewidth=2, markersize=5)

    # Add labels and title with enhanced formatting
    plt.xlabel('Distance', fontsize=14, labelpad=10)
    plt.ylabel('Cumulative Count of Valid Samples', fontsize=14, labelpad=10)
    plt.title('Cumulative Count of Valid Samples vs. Distance', fontsize=16, pad=15)

    # Customize grid style for better readability
    plt.grid(True, linestyle='--', linewidth=0.5)

    # Adjust layout for better spacing
    plt.tight_layout()

    # Display the plot
    plt.show()

In [ ]:
def find_border_points(model, final_output_tensor1, mal_x_batch2, iterations=10):
    """
    Finds the border points between two sets of samples using binary search and model predictions.

    Args:
        model (torch.nn.Module): The trained model used for inference.
        final_output_tensor1 (torch.Tensor): Tensor representing the starting points (point_a).
        mal_x_batch2 (torch.Tensor): Tensor representing the target points (point_b).
        iterations (int): Number of iterations for refining the border points.

    Returns:
        torch.Tensor: The refined border points.
        torch.Tensor: The distances between the border points and mal_x_batch2.
    """
    # Clone tensors to avoid modifying the originals
    point_a = final_output_tensor1.clone()  # Shape: (batch_size, features)
    point_b = mal_x_batch2.clone()          # Shape: (batch_size, features)

    # Compute the initial distances between point_a and point_b
    border_points = point_a.clone()
    closest_distances = torch.abs(point_a - mal_x_batch2).sum(dim=1)

    # Set the model to evaluation mode
    model.eval()

    # Iteratively refine border points
    for _ in range(iterations):
        # Compute midpoints between point_a and point_b
        samples = (point_a + point_b) / 2.0  # Shape: (batch_size, features)

        # Perform inference and get predictions
        with torch.no_grad():
            outputs = model(samples)
            predictions = outputs.argmax(dim=1)  # Shape: (batch_size,)

        # Create masks for valid and invalid samples
        valid_mask = (predictions == 0)

        # Update point_a and border_points where predictions are valid
        point_a[valid_mask] = samples[valid_mask]
        border_points[valid_mask] = samples[valid_mask]

        # Update point_b where predictions are invalid
        point_b[~valid_mask] = samples[~valid_mask]

        # Recalculate the distances between updated border points and mal_x_batch2
        closest_distances = torch.abs(border_points - mal_x_batch2).sum(dim=1)

    print(f"Sum of the distance between adversarial samples and malwares: {(torch.abs(final_output_tensor1 - mal_x_batch2)).sum().item():.2f}")
    print(f"Sum of the distance between border adversarial samples and malwares: {(torch.abs(border_points - mal_x_batch2)).sum().item():.2f}")
    return border_points, closest_distances

In [ ]:
import torch
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from torch.optim.lr_scheduler import OneCycleLR
from torch.optim.lr_scheduler import LambdaLR

def compare2(output):
    output = torch.argmax(output, dim=1)
    return output == 0

def compare(outputs, confidence):
    outputs[:,0] -= confidence
    output = torch.argmax(outputs, dim=1)
    return output == 0

def sin_func_torch(input_tensor):
    """
    A sine-based transformation applied to the input tensor.
    The function applies a sine transformation and rescales the output to [0, 1].
    """
    return (torch.sin((input_tensor - 0.5) * torch.pi) + 1) / 2.0


# Define the learning rate schedules (same as earlier)
def lr_linear(t, T, lr_start=1e-3, lr_end=0.9):
    return lr_start + t * (lr_end - lr_start) / T

def lr_exponential(t, T, lr_start=1e-3, lr_end=0.9):
    return lr_start * (lr_end / lr_start) ** (t / T)

def lr_cosine(t, T, lr_start=1e-3, lr_end=0.9):
    return lr_end + (lr_start - lr_end) * 0.5 * (1 + torch.cos(torch.tensor(np.pi * t / T)))

def lr_average(t, T, lr_start=1e-3, lr_end=0.9):
    return 0.5 * (lr_linear(t, T, lr_start, lr_end) + lr_exponential(t, T, lr_start, lr_end))

def lr_cosine_half(t, T, lr_start, lr_end):
    return lr_start + (lr_end - lr_start) * (1 - np.cos(np.pi * t /(2*T)))



def CW(model, imgs, y, mimic_output, removal_array, initial_mimicry, device='cuda', initial_const = 1., binary_search_steps=8, cons_increase_factor=10., LR_function=lr_exponential,
                 max_lr=2., min_lr=1e-4,update_step=1000, first_phase_iters=20000,second_phase_iters = 10000 , confidence=0.1, eps=1e-4):
    """
    Perform a Learning Rate Range Test for adversarial attack optimization using the Adam optimizer.

    - model: The target model to attack.
    - imgs: The original images to be attacked.
    - y: The true labels of the images.
    - removal_array: A mask array specifying which features are fixed.
    - mimicry: if the mimic_output is mimicry and integer not border , True.
    - device: The device on which to perform computations.
    - max_lr: Maximum learning rate to end with.
    - num_iter: Number of iterations to perform.
    - confidence: Confidence parameter for adversarial loss computation.
    """

    def grad_hook(grad):
        return grad * non_fixed_features_mask

    model.eval()

    # Preparing tensors
    imgs_tensor_original = imgs.clone().detach().to(dtype=torch.float32, device=device)
    imgs_tensor = torch.asin(2 * mimic_output - 1).to(device)
    labs_tensor = torch.zeros_like(y.squeeze(), dtype=torch.int64, device=device)

    batch_size = imgs.shape[0]

    # Create masks for 0s and 1s
    zero_mask = (imgs == 0)  # Mask where values are 0
    one_mask = (imgs == 1)   # Mask where values are 1

    # Add eps to 0s and subtract eps from 1s
    imgs_tensor[zero_mask] += eps
    imgs_tensor[one_mask] -= eps

    lower_bound = torch.zeros(batch_size, dtype=torch.float32, device=device)
    CONST = torch.full((batch_size,), initial_const, dtype=torch.float32, device=device)
    upper_bound = torch.full((batch_size,), 1e10, dtype=torch.float32, device=device)

    # Expand removal array to match the batch size
    expanded_removal_array = removal_array.expand(imgs.shape[0], -1)
    non_fixed_features_mask = torch.bitwise_or(expanded_removal_array, 1 - imgs.to(torch.uint8))

    # Initialize tracking variables
    l2dist = torch.sum((mimic_output - imgs_tensor_original) ** 2, dim=1)

    with torch.no_grad():
      outputs = model(mimic_output)
      o_success_mask = compare(outputs=outputs, confidence=0.)

      o_bestattack = torch.where(o_success_mask.unsqueeze(-1), mimic_output.clone().detach(), imgs_tensor_original.clone().detach()).to(device)
      o_bestl2 = torch.where(o_success_mask, l2dist.detach(), 1e10).to(device)

    #o_bestl2 = torch.full((batch_size,), 1e10, device=device)
    #o_bestattack = imgs.clone().detach()

    o_best_loss1 = torch.full((batch_size,), float('inf'), device=device)
    o_best_unsuccessfull_attack = imgs.clone().detach()
    #o_success_mask = torch.zeros(batch_size, dtype=torch.bool, device=device)



    # Total number of iterations
    total_iters = first_phase_iters + second_phase_iters

    for outer_step in range(binary_search_steps):
        print('------------------------------------------------------------------------------------------------------------------------------------')
        print(CONST[:20])

        modifier = torch.zeros_like(imgs_tensor, requires_grad=True).to(device)
        optimizer = torch.optim.Adam([modifier], lr=min_lr)
        modifier.register_hook(grad_hook)

        bestl2 = torch.full((batch_size,), 1e10, device=device)
        bestattack = imgs.clone().detach()
        best_modifier = torch.zeros_like(modifier).detach()  # Best modifier
        best_unsuccessfull_modifier = torch.zeros_like(modifier).detach()

        best_loss1 = torch.full((batch_size,), float('inf'), device=device)
        best_unsuccessfull_attack = imgs.clone().detach()
        success_mask = torch.zeros(batch_size, dtype=torch.bool, device=device)
        success_mask_inloop = torch.zeros(batch_size, dtype=torch.bool, device=device)

        for iteration in range(total_iters):

            # Phase 2: Fixed learning rate of 1e-3 (iterations 15,000 to 24,999)
            if iteration == first_phase_iters:
                print("************ second phase *************")
                output = model(bestattack)
                final_success = compare(output, confidence)
                final_output = torch.where(final_success.unsqueeze(-1), bestattack, best_unsuccessfull_attack)
                final_modifier = torch.where(final_success.unsqueeze(-1), best_modifier, best_unsuccessfull_modifier)

                #modifier = torch.tensor(final_modifier.clone().detach(), requires_grad=True).to(device)
                modifier = (final_modifier.clone().detach().requires_grad_(True)).to(device)
                optimizer = torch.optim.Adam([modifier], lr=2e-3)  # Use lr_min for 1000 iterations
                modifier.register_hook(grad_hook)

            if iteration < first_phase_iters:
                if iteration == 0 or (iteration + 1) % update_step == 0:
                    # Select your preferred learning rate schedule
                    current_lr = LR_function(iteration, first_phase_iters, min_lr, max_lr)  # You can switch to lr_linear, lr_exponential, or lr_cosine

                    # Update the optimizer's learning rate
                    for param_group in optimizer.param_groups:
                        param_group['lr'] = current_lr

            optimizer.zero_grad()

            # Attack Step: generate perturbed image
            modified_input = imgs_tensor + modifier
            newimg = torch.sin(modified_input) * 0.5 + 0.5
            output = model(newimg)

            l2dist = torch.sum((newimg - imgs_tensor_original) ** 2, dim=1)

            real = output[:, 0]
            other = output[:, 1]

            loss1 = torch.clamp((other - real + confidence), min=0.0)
            loss = (CONST * loss1) + l2dist

            total_loss = torch.sum(loss)
            total_loss.backward()
            optimizer.step()

            if iteration > 1100:
                with torch.no_grad():
                    success_mask_inloop = compare(output, confidence)
                    success_mask = torch.logical_or(success_mask_inloop, success_mask)
                    o_success_mask  = torch.logical_or(o_success_mask, success_mask_inloop)

                    update_mask = (l2dist < bestl2) & success_mask_inloop
                    bestl2 = torch.where(update_mask, l2dist, bestl2)
                    bestattack = torch.where(update_mask.unsqueeze(-1), newimg, bestattack)
                    best_modifier = torch.where(update_mask.unsqueeze(-1), modifier.detach(), best_modifier)

                    improved_mask = (loss1 < best_loss1)
                    best_loss1 = torch.where(improved_mask, loss1, o_best_loss1)
                    best_unsuccessfull_attack = torch.where(improved_mask.unsqueeze(-1), newimg, best_unsuccessfull_attack)
                    best_unsuccessfull_modifier = torch.where(improved_mask.unsqueeze(-1), modifier.detach(), best_unsuccessfull_modifier)

                    update_mask2 = (l2dist < o_bestl2) & success_mask_inloop
                    o_bestl2 = torch.where(update_mask2, l2dist, o_bestl2)
                    o_bestattack = torch.where(update_mask2.unsqueeze(-1), newimg, o_bestattack)

                    improved_mask2 = (loss1 < o_best_loss1)
                    o_best_loss1 = torch.where(improved_mask2, loss1, o_best_loss1)
                    o_best_unsuccessfull_attack = torch.where(improved_mask2.unsqueeze(-1), newimg, o_best_unsuccessfull_attack)


            if iteration == 0 or (iteration + 1) % 1000 == 0:
                current_lr = optimizer.param_groups[0]['lr']
                current_momentum, current_momentum2 = optimizer.param_groups[0]['betas']
                print(f"Iteration {iteration + 1:5}:\t Current LR = {current_lr:.4f} \t Current momentum = {current_momentum:.2f} \t Current success = {success_mask_inloop.sum().item()} \t success = {success_mask.sum().item()} \t Loss = {total_loss.item():.2f} \t l2dist = {torch.mean(bestl2[success_mask]).item():.4f}")

                print(f"Best results until now: \t success = {o_success_mask.sum().item()} \t l2dist = {torch.mean(o_bestl2[o_success_mask]).item():.4f}")

        failure_mask = ~success_mask_inloop
        upper_bound[success_mask_inloop] = torch.minimum(upper_bound[success_mask_inloop], CONST[success_mask_inloop])
        lower_bound[failure_mask] = torch.maximum(lower_bound[failure_mask], CONST[failure_mask])

        upper_limit_mask = upper_bound < 1e9
        CONST[upper_limit_mask] = (lower_bound[upper_limit_mask] + upper_bound[upper_limit_mask]) / 2
        CONST[failure_mask & ~upper_limit_mask] *= cons_increase_factor



    print('')
    output = model(o_bestattack)
    final_success = compare(output,0.)
    print(f"FINAL RESULT :\t #success = {final_success.sum().item()} \t total Loss = {total_loss.item():.2f}\t success Loss = {loss[final_success].sum().item():.2f} \t success l2dist = {torch.mean(o_bestl2[final_success]).item():.4f}")
    print('')


    final_output = torch.where(final_success.unsqueeze(-1), o_bestattack, o_best_unsuccessfull_attack)
    final_output_rounded = final_rounding(imgs, final_output, model, removal_array)

    output = model(final_output_rounded)
    final_success2 = compare(output,0.)
    cw_unsuccess = ~final_success2
    mimicry_success = compare(model(initial_mimicry), 0.)
    condition = (mimicry_success & cw_unsuccess)
    print('Number of updates with initial mimicry :',condition.sum().item())
    final_success_rounded = torch.where(condition.unsqueeze(-1), initial_mimicry, final_output_rounded)

    print('final number of success : ', compare(model(final_success_rounded),0.).sum().item())
    print('')
    results_rounded = evaluate_rounded_adversarial_examples(imgs, y, final_success_rounded, model, removal_array)
    print('')

    return final_output, results_rounded["adv_rounded"]

In [ ]:
import torch
import seaborn as sns
import matplotlib.pyplot as plt
from tqdm import tqdm

def plot_adv_distances(adv_rounded_list, labels, model, mal_x_batch2, device):
    """
    Plots the cumulative distance between adversarial tensors and a reference batch (mal_x_batch2).

    Parameters:
    - adv_rounded_list: List of adversarial tensors to be evaluated.
    - labels: List of labels for the adversarial tensors (for the plot legend).
    - model: The model used for predicting adversarial examples.
    - mal_x_batch2: The reference batch to compare adversarial tensors against.
    - device: Device to run the computations on (e.g., 'cuda' or 'cpu').
    - model_AT_rFGSM: A variant of the model to evaluate adversarial tensors.
    """

    # Set a style using Seaborn and select a color palette automatically
    sns.set(style='whitegrid')
    colors = sns.color_palette('husl', len(adv_rounded_list))  # Automatically select colors

    # Create the plot
    plt.figure(figsize=(20, 12))

    model.eval()

    # Iterate through each adv_rounded tensor and plot their distances
    for adv_rounded, color, label in tqdm(zip(adv_rounded_list, colors, labels),
                                          total=len(adv_rounded_list), desc="Processing Tensors"):
        # Pass the adv_rounded tensor through the model to get predictions
        adv_rounded = adv_rounded.to(device)  # Move tensor to the correct device
        outputs = model(adv_rounded)
        y_pred = outputs.argmax(dim=1)

        # Calculate the differences between adv_rounded and mal_x_batch2 for correctly classified samples
        mal_x_batch2 = mal_x_batch2.to(device)
        total_difference = torch.sum((adv_rounded[y_pred == 0] - mal_x_batch2[y_pred == 0]) ** 2, dim=1)

        # Sort the differences in ascending order
        sorted_differences, _ = torch.sort(total_difference)

        # Generate the x-axis (distance values) and y-axis (cumulative counts)
        x_values = sorted_differences.cpu()  # Move tensor back to CPU for plotting
        y_values = torch.arange(1, len(sorted_differences) + 1).cpu()  # Create a tensor for the y-axis values

        # Plot the line for the current adv_rounded tensor
        sns.lineplot(x=x_values, y=y_values, marker='o', color=color, linewidth=2, markersize=5, label=label)

    # Add labels and title with a bit more formatting
    plt.xlabel('Distance', fontsize=16, labelpad=10)
    plt.ylabel('Number of Valid Samples (Cumulative)', fontsize=16, labelpad=10)
    plt.title('Cumulative Count of Valid Samples vs. Distance', fontsize=18, pad=15)

    # Add legend for clarity
    plt.legend(title='Adversarial Tensors', fontsize=12, title_fontsize=14, loc='upper left')

    # Improve the look of the grid and remove spines for a cleaner look
    plt.grid(True, linestyle='--', linewidth=0.5)
    sns.despine()

    # Show the plot with tighter layout for better visualization
    plt.tight_layout()
    plt.show()

## Step 7 — Mimicry + CW against AT-rFGSM

### model_AT_rFGSM

In [ ]:
final_output_tensor1, final_output_tensor2, valid_distances_list = Mimicry_group_one(model_AT_rFGSM, removal_array, ben_x_batch, mal_x_batch2)

In [ ]:
combined_model.eval()
outputs = model_AT_rFGSM(final_output_tensor1)
y_pred = outputs.argmax(1)
print((y_pred == 0).sum().item())

In [ ]:
final_output_tensor1, final_output_tensor2, valid_distances_list = Mimicry(model_AT_rFGSM, removal_array, ben_x_batch, mal_x_batch2)

In [ ]:
results_rounded = evaluate_rounded_adversarial_examples(mal_x_batch2, mal_y_batch2, final_output_tensor1, model_AT_rFGSM, removal_array)

In [ ]:
print('the difference befroe final_rounding : ',torch.abs(mal_x_batch2 - final_output_tensor1).sum())
adv_rounded = final_rounding(mal_x_batch2, final_output_tensor1, model_AT_rFGSM, removal_array)
outputs = model_AT_rFGSM(adv_rounded)
y_pred = outputs.argmax(dim=1)
condition = y_pred.unsqueeze(-1).bool()
final_output_tensor1_new = torch.where(condition, final_output_tensor1, adv_rounded)
print('the difference befroe final_rounding : ', torch.abs(mal_x_batch2 - final_output_tensor1_new).sum())

outputs = model_AT_rFGSM(final_output_tensor1_new)
y_pred = outputs.argmax(1)
print((y_pred == 0).sum().item())

In [ ]:
# best confidence is equal to 0.4
adv_rounded_list = [final_output_tensor1, final_output_tensor1_new]  # Replace with your tensors
labels = ['Mimicry','Mimicry_improved']  # Labels for the legend

plot_adv_distances(adv_rounded_list, labels, model_AT_rFGSM, mal_x_batch2, device)

In [ ]:
border_points, closest_distances = find_border_points(model_AT_rFGSM, final_output_tensor1_new, mal_x_batch2, iterations=10)
closest_distances[:10]

In [ ]:
# Get the model's output and predicted class labels for each sample
outputs = model_AT_rFGSM(final_output_tensor2_new)
y_pred = outputs.argmax(dim=1)

# Ensure the condition tensor is in boolean format
condition = y_pred.unsqueeze(-1).bool()

# Use torch.where with the boolean condition to select between tensors
border_points2 = torch.where(condition, final_output_tensor2_new, border_points)

In [ ]:
a1_b1 = final_output_tensor1_new.clone()
a1_b2 = final_output_tensor2_new.clone()

a2_b1 = border_points.clone()
a2_b2 = border_points2.clone()

In [ ]:
# Define the best confidence level and mimicry settings
confidence = 0.4
mimicry_outputs = [a1_b1, a1_b2, a2_b1, a2_b2]
mimicry_outputs_names = ['a1_b1', 'a1_b2', 'a2_b1', 'a2_b2']

# Store results in lists
adv_list = []
adv_rounded_list = []

# Loop through each mimicry output, name, and flag to run the CW attack
for mimicry_output, name in zip(mimicry_outputs, mimicry_outputs_names):
    print('**********************************************************************************************************************************************')
    print(f"Mimicry Output: {name}")

    # Perform the CW attack with the current mimicry settings
    adv, adv_rounded = CW(model_AT_rFGSM, mal_x_batch2, mal_y_batch2,mimicry_output, removal_array, initial_mimicry=a1_b1, device='cuda', initial_const = 1., binary_search_steps=6, cons_increase_factor=10., LR_function=lr_cosine_half,
                          max_lr=1.2, min_lr=1e-3, update_step=500, first_phase_iters=10000,second_phase_iters = 2500 , confidence=confidence, eps=0.)

    # Append the results to the respective lists
    adv_list.append(adv)
    adv_rounded_list.append(adv_rounded)

In [ ]:
adv_rounded_list.append(a1_b1)
adv_rounded_list.append(final_output_tensor1)
labels = ['Mimicry + CW(a1_b1)', 'Mimicry + CW(a1_b2)', 'Mimicry + CW(a2_b1)', 'Mimicry + CW(a2_b2)', 'Mimicry_improved', 'Mimicry']  # Labels for the legend
plot_adv_distances(adv_rounded_list, labels, model_AT_rFGSM, mal_x_batch2, device)

In [ ]:
labels = ['Mimicry + CW(a1_b1)', 'Mimicry + CW(a2_b1)', 'Mimicry_improved', 'Mimicry']  # Labels for the legend
plot_adv_distances([adv_rounded_list[0],adv_rounded_list[2],adv_rounded_list[-2],adv_rounded_list[-1]], labels, model_AT_rFGSM, mal_x_batch2, device)

In [ ]:
adv_list_conf0, adv_list_conf0_rounded = CW(model_AT_rFGSM, mal_x_batch2, mal_y_batch2,a1_b1, removal_array, a1_b1, device='cuda', initial_const = 1., binary_search_steps=6, cons_increase_factor=10., LR_function=lr_cosine_half,
                                            max_lr=1.2, min_lr=1e-3, update_step=500, first_phase_iters=10000,second_phase_iters = 2500 , confidence=0., eps=0.)

In [ ]:
# best confidence is equal to 0.4
adv_rounded_list = [a1_b1, final_output_tensor1, adv_list_conf0, adv_list[0], adv_rounded_list[0]]  # Replace with your tensors
labels = ['Mimicry_improved', 'Mimicry','improved_mimicry_lowest_distance_l2','Mimicry + CW(a1_b1)_conf_0.4', 'Mimicry + CW(a1_b1)_rounded']  # Labels for the legend

plot_adv_distances(adv_rounded_list, labels, model_AT_rFGSM, mal_x_batch2, device)

## Step 8 — Mimicry + CW against combined_model

### combined_model

In [ ]:
final_output_tensor1, final_output_tensor2, valid_distances_list = Mimicry_group_one(combined_model, removal_array, ben_x_batch, mal_x_batch2)

In [ ]:
combined_model.eval()
outputs = combined_model(final_output_tensor1)
y_pred = outputs.argmax(1)
print((y_pred == 0).sum().item())

In [ ]:
final_output_tensor1, final_output_tensor2, valid_distances_list = Mimicry(combined_model, removal_array, ben_x_batch, mal_x_batch2)

In [ ]:
results_rounded = evaluate_rounded_adversarial_examples(mal_x_batch2, mal_y_batch2, final_output_tensor1, combined_model, removal_array)

In [ ]:
print('the difference befroe final_rounding : ',torch.abs(mal_x_batch2 - final_output_tensor1).sum())
adv_rounded = final_rounding(mal_x_batch2, final_output_tensor1, combined_model, removal_array)
outputs = combined_model(adv_rounded)
y_pred = outputs.argmax(dim=1)
condition = y_pred.unsqueeze(-1).bool()
final_output_tensor1_new = torch.where(condition, final_output_tensor1, adv_rounded)
print('the difference befroe final_rounding : ', torch.abs(mal_x_batch2 - final_output_tensor1_new).sum())

outputs = combined_model(final_output_tensor1_new)
y_pred = outputs.argmax(1)
print((y_pred == 0).sum().item())

In [ ]:
adv_rounded_list = [final_output_tensor1, final_output_tensor1_new]  # Replace with your tensors
labels = ['Mimicry','Mimicry_improved']  # Labels for the legend

plot_adv_distances(adv_rounded_list, labels, combined_model, mal_x_batch2, device)

In [ ]:
border_points, closest_distances = find_border_points(combined_model, final_output_tensor1_new, mal_x_batch2, iterations=10)
closest_distances[:10]

In [ ]:
# Get the model's output and predicted class labels for each sample
outputs = combined_model(final_output_tensor2_new)
y_pred = outputs.argmax(dim=1)

# Ensure the condition tensor is in boolean format
condition = y_pred.unsqueeze(-1).bool()

# Use torch.where with the boolean condition to select between tensors
border_points2 = torch.where(condition, final_output_tensor2_new, border_points)

In [ ]:
# Define the best confidence level and mimicry settings
confidence = 0.2
mimicry_outputs = [a1_b1, a1_b2, a2_b1, a2_b2]
mimicry_outputs_names = ['a1_b1', 'a1_b2', 'a2_b1', 'a2_b2']

# Store results in lists
adv_list = []
adv_rounded_list = []

# Loop through each mimicry output, name, and flag to run the CW attack
for mimicry_output, name in zip(mimicry_outputs, mimicry_outputs_names):
    print('**********************************************************************************************************************************************')
    print(f"Mimicry Output: {name}")

    # Perform the CW attack with the current mimicry settings
    adv, adv_rounded = CW(combined_model, mal_x_batch2, mal_y_batch2,mimicry_output, removal_array, initial_mimicry=a1_b1, device='cuda', initial_const = 10., binary_search_steps=6, cons_increase_factor=10., LR_function=lr_cosine_half,
                          max_lr=0.9, min_lr=1e-3, update_step=500, first_phase_iters=10000,second_phase_iters = 2500 , confidence=confidence, eps=0.)

    # Append the results to the respective lists
    adv_list.append(adv)
    adv_rounded_list.append(adv_rounded)

In [ ]:
adv_rounded_list.append(a1_b1)
adv_rounded_list.append(final_output_tensor1)
labels = ['Mimicry + CW(a1_b1)', 'Mimicry + CW(a1_b2)', 'Mimicry + CW(a2_b1)', 'Mimicry + CW(a2_b2)', 'Mimicry_improved', 'Mimicry']  # Labels for the legend
plot_adv_distances(adv_rounded_list, labels, combined_model, mal_x_batch2, device)

In [ ]:
labels = ['Mimicry + CW(a1_b1)', 'Mimicry + CW(a2_b1)', 'Mimicry_improved', 'Mimicry']  # Labels for the legend
plot_adv_distances([adv_rounded_list[0],adv_rounded_list[2],adv_rounded_list[-2],adv_rounded_list[-1]], labels, combined_model, mal_x_batch2, device)

In [ ]:
adv_list_conf0, adv_list_conf0_rounded = CW(combined_model, mal_x_batch2, mal_y_batch2,a1_b1, removal_array, a1_b1, device='cuda', initial_const = 10., binary_search_steps=6, cons_increase_factor=10., LR_function=lr_cosine_half,
                                            max_lr=0.9, min_lr=1e-3, update_step=500, first_phase_iters=10000,second_phase_iters = 2500 , confidence=0., eps=0.)

In [ ]:
# best confidence is equal to 0.2
adv_rounded_list1 = [a1_b1, final_output_tensor1, adv_list_conf0, adv_list[0], adv_rounded_list[0]]  # Replace with your tensors
labels = ['Mimicry_improved', 'Mimicry','improved_mimicry_lowest_distance_l2','Mimicry + CW(a1_b1)_conf_0.2', 'Mimicry + CW(a1_b1)_rounded']  # Labels for the legend

plot_adv_distances(adv_rounded_list1, labels, combined_model, mal_x_batch2, device)

In [ ]:
torch.abs(mal_x_batch2 - a1_b1).sum(1)[torch.abs(mal_x_batch2 - a1_b1).sum(1)>0][:200]

In [ ]:
torch.abs(mal_x_batch2 - adv_rounded_list[0]).sum(1)[torch.abs(mal_x_batch2 - a1_b1).sum(1)>0][:200]

## Step 9 — The mixed strategy: keep whichever worked, per sample

### New Section

In [ ]:
torch.abs(mal_x_batch2 - final_output_tensor1).sum()

In [ ]:
combined_model.eval()

print('the difference befroe final_rounding : ',torch.abs(mal_x_batch2 - final_output_tensor1).sum())
adv_rounded = final_rounding(mal_x_batch2, final_output_tensor1, combined_model, removal_array)
outputs = combined_model(adv_rounded)
y_pred = outputs.argmax(dim=1)
condition = y_pred.unsqueeze(-1).bool()
final_output_tensor1_new = torch.where(condition, final_output_tensor1, adv_rounded)
print('the difference befroe final_rounding : ', torch.abs(mal_x_batch2 - final_output_tensor1_new).sum())

outputs = combined_model(final_output_tensor1_new)
y_pred = outputs.argmax(1)
print((y_pred == 0).sum().item())

In [ ]:
equi_mimic = mal_x_batch2[y_pred==0]
equi_mimic.shape

In [ ]:
adv1, adv_rounded1 = CW(combined_model, mal_x_batch2, mal_y_batch2,final_output_tensor1_new, removal_array, initial_mimicry=final_output_tensor1_new, device='cuda', initial_const = 10., binary_search_steps=6, cons_increase_factor=10., LR_function=lr_cosine_half,
                          max_lr=0.9, min_lr=1e-3, update_step=500, first_phase_iters=10000,second_phase_iters = 2500 , confidence=0.2, eps=0.)

In [ ]:
labels = ['Mimicry','Mimicry_improved by cw_rounded','Mimicry_improved by cw']  # Labels for the legend
plot_adv_distances([final_output_tensor1[y_pred==0],adv_rounded1[y_pred==0],adv1[y_pred==0]], labels, combined_model, mal_x_batch2[y_pred==0], device)

In [ ]:
adv2, adv_rounded2 = CW(combined_model, equi_mimic, mal_y_batch2,equi_mimic, removal_array, initial_mimicry=equi_mimic, device='cuda', initial_const = 10., binary_search_steps=6, cons_increase_factor=10., LR_function=lr_cosine_half,
                          max_lr=0.9, min_lr=1e-3, update_step=500, first_phase_iters=10000,second_phase_iters = 2500 , confidence=0.2, eps=0.)

In [ ]:
final = adv_rounded1.clone()

In [ ]:
tensor1 = adv_rounded1[y_pred==0].clone()
tensor2 = adv_rounded2.clone()
initial_tensor = mal_x_batch2[y_pred==0]


# Assume tensor1, tensor2, and initial_tensor have the shape (samples, 10000)
# Assume combined_model is a model that takes an input of shape (samples, 10000) and returns a tensor.

samples, _ = tensor1.shape

# Get the predictions from the combined model for tensor1 and tensor2
predictions1 = combined_model(tensor1).argmax(dim=1)  # Shape: (samples,)
predictions2 = combined_model(tensor2).argmax(dim=1)  # Shape: (samples,)

# Create masks for where the predictions are equal to 0
mask1 = (predictions1 == 0)  # Shape: (samples,)
mask2 = (predictions2 == 0)  # Shape: (samples,)

# Compute the Euclidean distances between tensor1, tensor2, and the initial_tensor
# Using torch.norm with dim=1 for computing row-wise Euclidean distance.
distances1 = torch.norm(tensor1 - initial_tensor, dim=1)  # Shape: (samples,)
distances2 = torch.norm(tensor2 - initial_tensor, dim=1)  # Shape: (samples,)

# Both masks true means we need to pick the one with the smaller distance.
both_true = mask1 & mask2  # Shape: (samples,)

# Create selection masks:
# - select_tensor1: True where we pick from tensor1.
# - select_tensor2: True where we pick from tensor2.
select_tensor1 = mask1 & ~mask2 | (both_true & (distances1 <= distances2))
select_tensor2 = mask2 & ~mask1 | (both_true & (distances2 < distances1))

# Use the masks to create the final tensor.
# We need to broadcast the masks over the second dimension.
selected_samples = torch.where(
    select_tensor1.unsqueeze(1), tensor1, tensor2
)

final[y_pred==0] = selected_samples.clone()

In [ ]:
outputs = combined_model(final)
y_pred1 = outputs.argmax(dim=1)
(y_pred1==0).sum()

In [ ]:
torch.abs(mal_x_batch2[y_pred1==0] - final[y_pred1==0]).sum()

In [ ]:
cw_attack = torch.cat((adv_rounded1[y_pred==1].clone(), adv_rounded2.clone()), dim=0)
outputs = combined_model(cw_attack)
y_pred1 = outputs.argmax(dim=1)
(y_pred1==0).sum()

In [ ]:
cw_attack0, cw_attack = CW(combined_model, mal_x_batch2, mal_y_batch2,mal_x_batch2, removal_array, initial_mimicry=mal_x_batch2, device='cuda', initial_const = 10., binary_search_steps=6, cons_increase_factor=10., LR_function=lr_cosine_half,
                          max_lr=0.9, min_lr=1e-3, update_step=500, first_phase_iters=10000,second_phase_iters = 2500 , confidence=0.2, eps=0.)

In [ ]:
labels = ['CW', 'Mimicry + CW', 'MIX','Mimicry']  # Labels for the legend
plot_adv_distances([cw_attack,adv_rounded1,final,final_output_tensor1], labels, combined_model, mal_x_batch2, device)

## Step 10 — The L1 version

## Norm L1

In [ ]:
import torch
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from torch.optim.lr_scheduler import OneCycleLR
from torch.optim.lr_scheduler import LambdaLR

def compare(output):
    output = torch.argmax(output, dim=1)
    return output == 0

def sin_func_torch(input_tensor):
    """
    A sine-based transformation applied to the input tensor.
    The function applies a sine transformation and rescales the output to [0, 1].
    """
    return (torch.sin((input_tensor - 0.5) * torch.pi) + 1) / 2.0


# Define the learning rate schedules (same as earlier)
def lr_linear(t, T, lr_start=1e-3, lr_end=0.9):
    return lr_start + t * (lr_end - lr_start) / T

def lr_exponential(t, T, lr_start=1e-3, lr_end=0.9):
    return lr_start * (lr_end / lr_start) ** (t / T)

def lr_cosine(t, T, lr_start=1e-3, lr_end=0.9):
    return lr_end + (lr_start - lr_end) * 0.5 * (1 + torch.cos(torch.tensor(np.pi * t / T)))

def lr_average(t, T, lr_start=1e-3, lr_end=0.9):
    return 0.5 * (lr_linear(t, T, lr_start, lr_end) + lr_exponential(t, T, lr_start, lr_end))

def lr_cosine_half(t, T, lr_start, lr_end):
    return lr_start + (lr_end - lr_start) * (1 - np.cos(np.pi * t /(2*T)))



def CW(model, imgs, y, mimic_output, removal_array, device='cuda', initial_const = 1., binary_search_steps=8, cons_increase_factor=10., LR_function=lr_exponential,
                 max_lr=2., min_lr=1e-4,update_step=1000, first_phase_iters=20000,second_phase_iters = 10000 , confidence=0.1, eps=1e-4):
    """
    Perform a Learning Rate Range Test for adversarial attack optimization using the Adam optimizer.

    - model: The target model to attack.
    - imgs: The original images to be attacked.
    - y: The true labels of the images.
    - removal_array: A mask array specifying which features are fixed.
    - device: The device on which to perform computations.
    - max_lr: Maximum learning rate to end with.
    - num_iter: Number of iterations to perform.
    - confidence: Confidence parameter for adversarial loss computation.
    """

    def grad_hook(grad):
        return grad * non_fixed_features_mask

    model.eval()

    # Preparing tensors
    imgs_tensor_original = imgs.clone().detach().to(dtype=torch.float32, device=device)
    imgs_tensor = torch.asin(2 * mimic_output - 1).to(device)
    labs_tensor = torch.zeros_like(y.squeeze(), dtype=torch.int64, device=device)

    batch_size = imgs.shape[0]

    # Create masks for 0s and 1s
    zero_mask = (imgs == 0)  # Mask where values are 0
    one_mask = (imgs == 1)   # Mask where values are 1

    # Add eps to 0s and subtract eps from 1s
    imgs_tensor[zero_mask] += eps
    imgs_tensor[one_mask] -= eps

    lower_bound = torch.zeros(batch_size, dtype=torch.float32, device=device)
    CONST = torch.full((batch_size,), initial_const, dtype=torch.float32, device=device)
    upper_bound = torch.full((batch_size,), 1e10, dtype=torch.float32, device=device)

    # Expand removal array to match the batch size
    expanded_removal_array = removal_array.expand(imgs.shape[0], -1)
    non_fixed_features_mask = torch.bitwise_or(expanded_removal_array, 1 - imgs.to(torch.uint8))


    # Initialize tracking variables
    with torch.no_grad():
      outputs = model(mimic_output)
      y_pred = outputs.argmax(1)
      l2dist = torch.sum((mimic_output - imgs_tensor_original) ** 2, dim=1)

      o_bestattack = torch.where(y_pred.unsqueeze(-1).to(torch.uint8), imgs_tensor_original.clone().detach(), mimic_output.clone().detach()).to(device)
      o_bestl2 = torch.where(y_pred.to(torch.uint8), 1e10, l2dist.detach()).to(device)
      o_success_mask = torch.where(y_pred.to(torch.uint8), 0, 1).to(device)

    #o_bestl2 = torch.full((batch_size,), 1e10, device=device)
    #o_bestattack = imgs.clone().detach()

    o_best_loss1 = torch.full((batch_size,), float('inf'), device=device)
    o_best_unsuccessfull_attack = imgs.clone().detach()
    #o_success_mask = torch.zeros(batch_size, dtype=torch.bool, device=device)



    # Total number of iterations
    total_iters = first_phase_iters + second_phase_iters

    for outer_step in range(binary_search_steps):
        print('------------------------------------------------------------------------------------------------------------------------------------')
        print(CONST)

        modifier = torch.zeros_like(imgs_tensor, requires_grad=True).to(device)
        optimizer = torch.optim.Adam([modifier], lr=min_lr)
        modifier.register_hook(grad_hook)

        bestl2 = torch.full((batch_size,), 1e10, device=device)
        bestattack = imgs.clone().detach()
        best_modifier = torch.zeros_like(modifier).detach()  # Best modifier
        best_unsuccessfull_modifier = torch.zeros_like(modifier).detach()

        best_loss1 = torch.full((batch_size,), float('inf'), device=device)
        best_unsuccessfull_attack = imgs.clone().detach()
        success_mask = torch.zeros(batch_size, dtype=torch.bool, device=device)
        success_mask_inloop = torch.zeros(batch_size, dtype=torch.bool, device=device)

        for iteration in range(total_iters):

            # Phase 2: Fixed learning rate of 1e-3 (iterations 15,000 to 24,999)
            if iteration == first_phase_iters:
                print("************ second phase *************")
                output = model(bestattack)
                final_success = compare(output)
                final_output = torch.where(final_success.unsqueeze(-1), bestattack, best_unsuccessfull_attack)
                final_modifier = torch.where(final_success.unsqueeze(-1), best_modifier, best_unsuccessfull_modifier)

                modifier = torch.tensor(final_modifier.clone().detach(), requires_grad=True).to(device)
                optimizer = torch.optim.Adam([modifier], lr=1e-3)  # Use lr_min for 1000 iterations
                modifier.register_hook(grad_hook)

            if iteration < first_phase_iters:
                if iteration == 0 or (iteration + 1) % update_step == 0:
                    # Select your preferred learning rate schedule
                    current_lr = LR_function(iteration, first_phase_iters, min_lr, max_lr)  # You can switch to lr_linear, lr_exponential, or lr_cosine

                    # Update the optimizer's learning rate
                    for param_group in optimizer.param_groups:
                        param_group['lr'] = current_lr

            optimizer.zero_grad()

            # Attack Step: generate perturbed image
            modified_input = imgs_tensor + modifier
            newimg = torch.sin(modified_input) * 0.5 + 0.5
            output = model(newimg)

            #l2dist = torch.sum((newimg - imgs_tensor_original) ** 2, dim=1)
            perturbation = torch.where((imgs_tensor_original==1.), (imgs_tensor_original - newimg), (newimg - imgs_tensor_original))
            l2dist = torch.sum(perturbation, dim=1)

            real = output[:, 0]
            other = output[:, 1]

            loss1 = torch.clamp((other - real + confidence), min=0.0)
            loss = (CONST * loss1) + l2dist

            total_loss = torch.sum(loss)
            total_loss.backward()
            optimizer.step()

            if iteration > 1100:
                with torch.no_grad():
                    success_mask_inloop = compare(output)
                    success_mask = torch.logical_or(success_mask_inloop, success_mask)
                    o_success_mask  = torch.logical_or(o_success_mask, success_mask_inloop)

                    update_mask = (l2dist < bestl2) & success_mask_inloop
                    bestl2 = torch.where(update_mask, l2dist, bestl2)
                    bestattack = torch.where(update_mask.unsqueeze(-1), newimg, bestattack)
                    best_modifier = torch.where(update_mask.unsqueeze(-1), modifier.detach(), best_modifier)

                    improved_mask = (loss1 < best_loss1)
                    best_loss1 = torch.where(improved_mask, loss1, o_best_loss1)
                    best_unsuccessfull_attack = torch.where(improved_mask.unsqueeze(-1), newimg, best_unsuccessfull_attack)
                    best_unsuccessfull_modifier = torch.where(improved_mask.unsqueeze(-1), modifier.detach(), best_unsuccessfull_modifier)

                    update_mask2 = (l2dist < o_bestl2) & success_mask_inloop
                    o_bestl2 = torch.where(update_mask2, l2dist, o_bestl2)
                    o_bestattack = torch.where(update_mask2.unsqueeze(-1), newimg, o_bestattack)

                    improved_mask2 = (loss1 < o_best_loss1)
                    o_best_loss1 = torch.where(improved_mask2, loss1, o_best_loss1)
                    o_best_unsuccessfull_attack = torch.where(improved_mask2.unsqueeze(-1), newimg, o_best_unsuccessfull_attack)


            if iteration == 0 or (iteration + 1) % 100 == 0:
                current_lr = optimizer.param_groups[0]['lr']
                current_momentum, current_momentum2 = optimizer.param_groups[0]['betas']
                print(f"Iteration {iteration + 1:5}:\t Current LR = {current_lr:.4f} \t Current momentum = {current_momentum:.2f} \t Current success = {success_mask_inloop.sum().item()} \t success = {success_mask.sum().item()} \t Loss = {total_loss.item():.4f} \t l2dist = {torch.mean(bestl2[success_mask]).item():.4f}")

                print(f"Best results until now: \t success = {o_success_mask.sum().item()} \t l2dist = {torch.mean(o_bestl2[o_success_mask]).item():.4f}")

        failure_mask = ~success_mask_inloop
        upper_bound[success_mask_inloop] = torch.minimum(upper_bound[success_mask_inloop], CONST[success_mask_inloop])
        lower_bound[failure_mask] = torch.maximum(lower_bound[failure_mask], CONST[failure_mask])

        upper_limit_mask = upper_bound < 1e9
        CONST[upper_limit_mask] = (lower_bound[upper_limit_mask] + upper_bound[upper_limit_mask]) / 2
        CONST[failure_mask & ~upper_limit_mask] *= cons_increase_factor



    print('')
    output = model(o_bestattack)
    final_success = compare(output)
    print(f"FINAL RESULT :\t #success = {final_success.sum().item()} \t total Loss = {total_loss.item():.4f}\t success Loss = {loss[final_success].sum().item():.4f} \t success l2dist = {torch.mean(o_bestl2[final_success]).item():.4f}")
    print('')

    final_output = torch.where(final_success.unsqueeze(-1), o_bestattack, o_best_unsuccessfull_attack)
    results_rounded = evaluate_rounded_adversarial_examples(imgs, y, final_output, combined_model, removal_array)
    print('')

    return final_output, results_rounded["adv_rounded"]

In [ ]:
adv, adv_rounded = CW(combined_model, mal_x_batch2, mal_y_batch2,a1_b1, removal_array, device='cuda', initial_const = 1., binary_search_steps=8, cons_increase_factor=10., LR_function=lr_cosine_half,
                 max_lr=1.5, min_lr=1e-3, update_step=500, first_phase_iters=15000,second_phase_iters = 5000 , confidence=0.1, eps=0.)

In [ ]:
results_fixed_elements = ensure_fixed_elements_unchanged(mal_x_batch2, adv, adv_rounded, removal_array)

In [ ]:
import torch
import seaborn as sns
import matplotlib.pyplot as plt

# Example list of adv_rounded tensors to be plotted
adv_rounded_list = [a1_b1, adv_rounded]  # Replace with your tensors
colors = ['b', 'r']  # Different colors for each plot
labels = ['Mimicry', 'Mimicry + CW(a1_b1)']  # Labels for the legend

# Set a style using Seaborn
sns.set(style='whitegrid')

# Create the plot
plt.figure(figsize=(20, 12))

# Iterate through each adv_rounded tensor and plot their distances
for adv_rounded, color, label in zip(adv_rounded_list, colors, labels):
    # Pass the adv_rounded tensor through the model to get predictions
    outputs = combined_model(adv_rounded)
    y_pred = outputs.argmax(dim=1)

    # Calculate the differences between adv_rounded and mal_x_batch2 for correctly classified samples
    total_difference = torch.abs(adv_rounded[y_pred == 0] - mal_x_batch2[y_pred == 0]).sum(1)

    # Sort the differences in ascending order
    sorted_differences, _ = torch.sort(total_difference)

    # Generate the x-axis (distance values) and y-axis (cumulative counts)
    x_values = sorted_differences.cpu()
    y_values = list(range(1, len(sorted_differences) + 1))

    # Plot the line for the current adv_rounded tensor
    sns.lineplot(x=x_values, y=y_values, marker='o', color=color, linewidth=2, markersize=5, label=label)

# Add labels and title with a bit more formatting
plt.xlabel('Distance', fontsize=14, labelpad=10)
plt.ylabel('Number of Valid Samples (Cumulative)', fontsize=14, labelpad=10)
plt.title('Cumulative Count of Valid Samples vs. Distance', fontsize=16, pad=15)

# Add legend for clarity
plt.legend(title='Adversarial Tensors', fontsize=12)

# Improve the look of the grid
plt.grid(True, linestyle='--', linewidth=0.5)

# Show the plot with tighter layout
plt.tight_layout()
plt.show()